In [1]:

!pip -q install pandas numpy scikit-learn scipy

In [2]:

import pandas as pd
import numpy as np

from google.colab import files

uploaded = files.upload()
file_path = next(iter(uploaded))

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

Saving lmp_final - lmp_semantic_review(1).csv to lmp_final - lmp_semantic_review(1) (1).csv
Dataset shape: (357, 8)

Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment']

First 5 rows:


,ID,Original,Language,Domain,Source,Simplified,LMP score,Human_Assessment
0,en_tos_001,"As a result, these Terms of Service help defin...",en,tos_privacy,google_tos,"Because of this, these Terms of Service help d...",10,Legal meaning and key details preserved — Key ...
1,en_tos_002,"Our services are designed to work together, ma...",en,tos_privacy,google_tos,Our services are designed to work together so ...,9,Legal meaning and key details preserved — Key ...
2,en_tos_003,"We also make various policies, help centers, a...",en,tos_privacy,google_tos,"We also provide various policies, help centers...",9,Legal meaning and key details preserved — Key ...
3,en_tos_004,"comply with applicable laws, including export ...",en,tos_privacy,google_tos,"Comply with applicable laws, including export ...",10,Legal meaning and key details preserved — Key ...
4,en_tos_005,using our services (including the content they...,en,tos_privacy,google_tos,Do not use our services (including the content...,10,Legal meaning and key details preserved — Key ...


In [5]:
# Rename the score column to the name our experiment code expects
df = df.rename(columns={"LMP score": "LMP"})

required_columns = ["Original", "Simplified", "LMP"]
missing = [col for col in required_columns if col not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["LMP"] = pd.to_numeric(df["LMP"], errors="coerce")

print("Dataset shape:", df.shape)
print("Missing scores:", df["LMP"].isna().sum())
print("Score distribution:")
print(df["LMP"].value_counts().sort_index())

Dataset shape: (357, 8)
Missing scores: 0
Score distribution:
LMP
4       4
5       5
6      17
7      34
8      71
9     102
10    124
Name: count, dtype: int64


In [36]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr
import numpy as np
import pandas as pd

# Use the exact dataset and split used by the XLM-R experiment
baseline_df = model_df.reset_index(drop=True).copy()

all_idx = np.arange(len(baseline_df))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42
)

# Match the original two-stage split: only train_part_idx is training data
train_part_idx, val_idx = train_test_split(
    train_idx, test_size=0.15, random_state=42
)

train_df = baseline_df.iloc[train_part_idx].copy()
test_df = baseline_df.iloc[test_idx].copy()

# Fit vocabulary and IDF weights on training sentences ONLY
train_sentences = (
    train_df["Original"].astype(str).tolist()
    + train_df["Simplified"].astype(str).tolist()
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000
)
vectorizer.fit(train_sentences)

# Transform train and test pairs using the training vocabulary
def pair_cosines(frame):
    originals = vectorizer.transform(frame["Original"].astype(str))
    simplified = vectorizer.transform(frame["Simplified"].astype(str))
    return np.asarray(
        originals.multiply(simplified).sum(axis=1)
    ).ravel()

train_cosine = pair_cosines(train_df)
test_cosine = pair_cosines(test_df)

y_train = train_df["LMP"].astype(float).to_numpy()
y_test = test_df["LMP"].astype(float).to_numpy()

# Calibrate using training data only
calibrator = LinearRegression()
calibrator.fit(train_cosine.reshape(-1, 1), y_train)

predictions = np.clip(
    calibrator.predict(test_cosine.reshape(-1, 1)), 1, 10
)

print("LEAKAGE-FREE TF-IDF BASELINE")
print("-" * 45)
print(f"Training pairs: {len(train_df)}")
print(f"Validation pairs (unused by baseline): {len(val_idx)}")
print(f"Test pairs: {len(test_df)}")
print(f"Pearson (cosine vs LMP): {pearsonr(test_cosine, y_test).statistic:.4f}")
print(f"Spearman (cosine vs LMP): {spearmanr(test_cosine, y_test).statistic:.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, predictions):.4f}")

tfidf_results = test_df[
    ["ID", "Original", "Simplified", "Language", "Domain", "LMP"]
].copy()
tfidf_results["TFIDF_Cosine"] = test_cosine
tfidf_results["Predicted_LMP"] = predictions
display(tfidf_results.head(10))

tfidf_results.to_csv("/content/tfidf_clean_test_predictions.csv", index=False)

LEAKAGE-FREE TF-IDF BASELINE
---------------------------------------------
Training pairs: 242
Validation pairs (unused by baseline): 43
Test pairs: 72
Pearson (cosine vs LMP): 0.6034
Spearman (cosine vs LMP): 0.7085
Test RMSE: 1.1634
Test MAE: 0.7924


,ID,Original,Simplified,Language,Domain,LMP,TFIDF_Cosine,Predicted_LMP
222,hi_tos_018,दूसरों को या खुद को नुकसान न पहुंचाना या दूसरों के साथ या अपने साथ बुरा बर्ताव न करना (या न तो ऐसे नुकसान या बुरे बर्ताव की धमकी देना या इन्हें बढ़ावा देना).,दूसरों को या खुद को नुकसान न पहुंचाना और बुरा बर्ताव न करना।,hi,tos_privacy,6.0,0.782214,9.167916
42,en_tos_043,"If we are unable to resolve a dispute within sixty (60) days after the Notice of Dispute is received, you or we may commence arbitration.","If we cannot resolve a dispute within 60 days after receiving the Notice of Dispute, you or we can start arbitration.",en,tos_privacy,9.0,0.807597,9.276008
291,hi_rental_006,अनुबंध समाप्त करने के लिए \_\_\_\_\_\_ पूर्व लिखित सूचना देनी होगी।,अनुबंध समाप्त करने के लिए पूर्व लिखित सूचना देनी होगी।,hi,rental,10.0,1.000000,10.000000
300,hi_rental_015,बातचीत से समाधान न होने पर \_\_\_\_\_\_ के सक्षम न्यायालय में कार्यवाही की जा सकती है।,बातचीत से समाधान न होने पर कार्यवाही की जा सकती है \_\_\_\_\_\_ के सक्षम न्यायालय में।,hi,rental,8.0,0.937295,9.828324
56,en_tos_057,These business or user terms and conditions are available on the respective Participating Platforms.,These terms and conditions are available on the respective Participating Platforms.,en,tos_privacy,9.0,0.862690,9.510620
335,hi_rental_050,इसलिए अब प्रथम पक्ष व द्वितीय पक्ष उक्त किरायानामा दिनांक ————— तक के लिये निष्पादित करते हैं।,इसलिए अब प्रथम पक्ष और द्वितीय पक्ष किराया तक दिनांक ————— तक देते हैं।,hi,rental,8.0,0.612261,8.444175
15,en_tos_016,"Regardless of where your information is processed, we apply the same protections described in this policy.","Regardless of processing location, we apply the same protections as described in this policy.",en,tos_privacy,9.0,0.735272,8.968015
57,en_tos_058,"For the respective contents of the Participating Platforms, the respective affiliate of Paytm is solely responsible only for those content that it actually operates.","For the content of the Participating Platforms, the relevant Paytm affiliate is responsible only for the content it operates.",en,tos_privacy,9.0,0.606212,8.418418
245,hi_tos_041,"ऐसा होने पर या इसकी वजह से, कोई तीसरा पक्ष कानूनी कार्रवाइयां (इसमें सरकारी अधिकारियों की कार्रवाइयां शामिल हैं) कर सकता है.","ऐसा होने पर या इसकी वजह से, कोई तीसरा पक्ष कानूनी कार्रवाए कर सकता है, इसमें सरकारी अधिकारियों की कार्रवाए भी शामिल हैं.",hi,tos_privacy,9.0,0.918042,9.746337
124,en_tos_125,We decide how long we need information on a case-by-case basis. Here's what we consider when we decide:,We decide how long to keep information on a case-by-case basis. We consider these factors:,en,tos_privacy,9.0,0.729535,8.943584


In [7]:

!pip -q install sentence-transformers scipy scikit-learn

In [8]:

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Keep only complete pairs with valid reference scores
data = df[["Original", "Simplified", "LMP"]].dropna().copy()
data["LMP"] = pd.to_numeric(data["LMP"], errors="coerce")
data = data.dropna(subset=["LMP"])
data = data[data["LMP"].between(1, 10)].reset_index(drop=True)

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

original_embeddings = model.encode(
    data["Original"].astype(str).tolist(),
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

simplified_embeddings = model.encode(
    data["Simplified"].astype(str).tolist(),
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

data["Semantic_Cosine"] = np.sum(
    original_embeddings * simplified_embeddings,
    axis=1
)

print("Sentence pairs:", len(data))
print("\nSemantic similarity summary:")
print(data["Semantic_Cosine"].describe())
display(data[["Original", "Simplified", "LMP", "Semantic_Cosine"]].head())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

Sentence pairs: 357

Semantic similarity summary:
count    357.000000
mean       0.933004
std        0.069801
min        0.539643
25%        0.906406
50%        0.956029
75%        0.981200
max        1.000000
Name: Semantic_Cosine, dtype: float64


,Original,Simplified,LMP,Semantic_Cosine
0,"As a result, these Terms of Service help defin...","Because of this, these Terms of Service help d...",10,0.997033
1,"Our services are designed to work together, ma...",Our services are designed to work together so ...,9,0.958247
2,"We also make various policies, help centers, a...","We also provide various policies, help centers...",9,0.956029
3,"comply with applicable laws, including export ...","Comply with applicable laws, including export ...",10,0.971900
4,using our services (including the content they...,Do not use our services (including the content...,10,0.814186


In [9]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr
import numpy as np

indices = np.arange(len(data))
train_idx, test_idx = train_test_split(
    indices, test_size=0.20, random_state=42
)

X = data[["Semantic_Cosine"]].values
y = data["LMP"].values

calibrator = LinearRegression()
calibrator.fit(X[train_idx], y[train_idx])

predictions = np.clip(calibrator.predict(X[test_idx]), 1, 10)
y_test = y[test_idx]
cosine_test = X[test_idx, 0]

print("BASELINE 2: MULTILINGUAL SENTENCE TRANSFORMER")
print("-" * 50)
print(f"Test pairs: {len(test_idx)}")
print(f"Pearson correlation: {pearsonr(cosine_test, y_test).statistic:.4f}")
print(f"Spearman correlation: {spearmanr(cosine_test, y_test).statistic:.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, predictions):.4f}")

BASELINE 2: MULTILINGUAL SENTENCE TRANSFORMER
--------------------------------------------------
Test pairs: 72
Pearson correlation: 0.5875
Spearman correlation: 0.6573
Test RMSE: 1.1020
Test MAE: 0.8461


In [11]:

!git clone https://github.com/GRAAL-Research/JUDGEBERT.git
%cd JUDGEBERT
!find article_src -maxdepth 3 -type f

Cloning into 'JUDGEBERT'...
remote: Enumerating objects: 1096, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 1096 (delta 24), reused 45 (delta 9), pack-reused 981 (from 1)
Receiving objects: 100% (1096/1096), 33.71 MiB | 11.32 MiB/s, done.
Resolving deltas: 100% (678/678), done.
Updating files: 100% (98/98), done.
/content/JUDGEBERT
article_src/dataset_analysis/dataset_analysis.py
article_src/dataset_analysis/__init__.py
article_src/dataset_analysis/dataset_analysis_eur-lex-sum_fr.py
article_src/dataset_analysis/dataset_analysis_fr.py
article_src/src/generate_dataset_only_requirements.txt
article_src/src/tools.py
article_src/src/sweep_eval.sh
article_src/src/few_shot_training.py
article_src/src/sweep_flaubert_base_da.sh
article_src/src/models.py
article_src/src/evaluate_tools.py
article_src/src/__init__.py
article_src/src/evaluation_llm.py
article_src/src/training_only_requirements.txt
article_src/src/metrics/metr

In [12]:

%cd /content/JUDGEBERT
!sed -n '1,240p' article_src/src/models.py

/content/JUDGEBERT
# All LLM we want to evaluate
model_names = [
    "jpacifico/Chocolatine-14B-Instruct-DPO-v1.2",
    "jpacifico/Chocolatine-14B-Instruct-DPO-v1.2",
    "jpacifico/French-Alpaca-Llama3-8B-Instruct-v1.0",
    "OpenLLM-France/Lucie-7B",
    "OpenLLM-France/Lucie-7B-Instruct",
    "OpenLLM-France/Lucie-7B-Instruct-human-data",
    "OpenLLM-France/Claire-7B-FR-Instruct-0.1",
    "unsloth/llama-3-8B-bnb-4bit",
    "unsloth/llama-3-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-1B-bnb-4bit",
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    "bigscience/bloom-560m",
    "bigscience/bloom-1b1",
    "bigscience/bloom-1b7",
    "bigscience/bloom-7b1",
    "bigscience/bloomz-560m",
    "bigscience/bloomz-1b1",
    "unsloth/Qwen2.5-0.5B-bnb-4bit",
    "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    "unsloth/Qw

In [13]:

!sed -n '1,240p' article_src/src/factory_model_tokenizer.py
!sed -n '1,240p' article_src/src/factory_dataset.py

from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    BitsAndBytesConfig,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
)
from transformers import logging

from unsloth import FastLanguageModel

logging.set_verbosity_warning()


def model_tokenizer_factory(model_name, device, token, seed: int = 42):
    if "gpt" in model_name:
        model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
        tokenizer = GPT2TokenizerFast.from_pretrained(model_name)

        # To handle padding when prompting
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"
    elif "bert" in model_name.lower():
        # BERT based model (along with RoBERTa).
        model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
        model.eval()
        tokenizer = AutoTokenizer.from_pretrained(model_name)
    elif (
        "bloom" in model_name.lower()
        or "moe" in model_name.

In [14]:
%cd /content/JUDGEBERT
!sed -n '1,260p' article_src/src/few_shot_training.py

/content/JUDGEBERT
import argparse
import logging
import os
import pickle
import shutil

import wandb
from poutyne import set_seeds
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    EarlyStoppingCallback,
)

from metrics.metrics import (
    compute_metrics,
    eval_compute_metrics_identical,
    eval_compute_metrics_unrelated,
)
from factory_dataset import dataset_factory
from tools import (
    bool_parse,
)

log = logging.getLogger("pytorch_lightning")
log.propagate = False
log.setLevel(logging.ERROR)

# Boilerplate params
os.environ["WANDB_PROJECT"] = "KILL"  # To control the Wandb project name in HuggingFace

parser = argparse.ArgumentParser()
parser.add_argument(
    "--seed",
    type=int,
    default=42,
    help="The seed to use for training.",
)

parser.add_argument(
    "--root",
    type=str,
    default=".",
    help="Root directory.",
)

parser.add_argument(
    "--da

In [15]:

!sed -n '1,240p' article_src/src/evaluation_loop.py
!sed -n '1,240p' article_src/src/tools.py

import os
from functools import partial

import numpy as np
import torch
import wandb
from dotenv import dotenv_values

from factory_dataset import dataset_factory
from evaluate_tools import evaluation_llm_prompting
from factory_model_tokenizer import model_tokenizer_factory
from metrics.metrics import (
    compute_metrics,
    eval_compute_metrics_identical,
    eval_compute_metrics_unrelated,
)

# To make Wandb silent
os.environ["WANDB_SILENT"] = "true"


def evaluation_loop(
    model_name: str,
    data_augmentation: bool,
):
    params = {"batched": True, "batch_size": 16}
    output_dir = "predictions"
    os.makedirs(output_dir, exist_ok=True)

    secrets = dotenv_values(".env")

    huggingface_token = secrets["huggingface_token"]

    device = torch.device("cuda")

    clean_model_name = model_name.split("/")[-1]

    wandb.init(
        project=f"kil_llm_eval",
        config={"model_name": model_name, "data_augmentation": data_augmentation},
        name=f"{clean_model_nam

In [16]:

%cd /content/JUDGEBERT
!grep -n -E "checkpoint|TrainingArguments|tokenize|compute_metrics|label|num_epoch|learning_rate|def main|def train" article_src/src/few_shot_training.py

/content/JUDGEBERT
11:    TrainingArguments,
19:    compute_metrics,
20:    eval_compute_metrics_identical,
21:    eval_compute_metrics_unrelated,
68:    help="The model name to use as the starting model checkpoint.",
76:num_epoch = args.epochs
86:checkpoint = args.model_name
87:tokenizer = AutoTokenizer.from_pretrained(checkpoint)
90:def tokenize_function(example):
91:    return tokenizer(
96:tokenized_legal_annotation_dataset = legal_precision_annotation_dataset.map(
97:    tokenize_function, batched=True
99:tokenize_holdout_identical_dataset = holdout_identical_dataset.map(
100:    tokenize_function, batched=True
102:tokenize_holdout_unrelated_dataset = holdout_unrelated_dataset.map(
103:    tokenize_function, batched=True
105:data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
107:output_dir = f"legal_meaning_bert_train_{checkpoint}_WithDA_{data_augmentation}_{seed}"
108:training_args = TrainingArguments(
114:    num_train_epochs=num_epoch,
115:    save_total_limit=num_epo

In [17]:

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nLanguage distribution:")
print(df["Language"].value_counts(dropna=False))
print("\nDomain distribution:")
print(df["Domain"].value_counts(dropna=False))
print("\nLMP score distribution:")
print(df["LMP"].value_counts(dropna=False).sort_index())
print("\nMissing values:")
print(df[["Original", "Simplified", "Language", "Domain", "LMP"]].isna().sum())

Dataset shape: (357, 8)
Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP', 'Human_Assessment']

Language distribution:
Language
en    205
hi    152
Name: count, dtype: int64

Domain distribution:
Domain
tos_privacy    227
rental         130
Name: count, dtype: int64

LMP score distribution:
LMP
4       4
5       5
6      17
7      34
8      71
9     102
10    124
Name: count, dtype: int64

Missing values:
Original      0
Simplified    0
Language      0
Domain        0
LMP           0
dtype: int64


In [18]:

from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# Keep the required columns and clean the labels
model_df = df[
    ["ID", "Original", "Simplified", "Language", "Domain", "LMP"]
].copy()

model_df["LMP"] = pd.to_numeric(model_df["LMP"], errors="coerce")
model_df = model_df.dropna(
    subset=["Original", "Simplified", "Language", "Domain", "LMP"]
).reset_index(drop=True)

model_df["LMP"] = model_df["LMP"].astype(float)

# Reproducible 80/20 split
indices = np.arange(len(model_df))
train_idx, test_idx = train_test_split(
    indices, test_size=0.20, random_state=42
)

train_df = model_df.iloc[train_idx].reset_index(drop=True)
test_df = model_df.iloc[test_idx].reset_index(drop=True)

print("Training pairs:", len(train_df))
print("Test pairs:", len(test_df))
print("\nTraining language counts:")
print(train_df["Language"].value_counts())
print("\nTest language counts:")
print(test_df["Language"].value_counts())
print("\nTraining domain counts:")
print(train_df["Domain"].value_counts())
print("\nTest domain counts:")
print(test_df["Domain"].value_counts())

Training pairs: 285
Test pairs: 72

Training language counts:
Language
en    164
hi    121
Name: count, dtype: int64

Test language counts:
Language
en    41
hi    31
Name: count, dtype: int64

Training domain counts:
Domain
tos_privacy    177
rental         108
Name: count, dtype: int64

Test domain counts:
Domain
tos_privacy    50
rental         22
Name: count, dtype: int64


In [19]:

!pip -q install transformers accelerate

In [20]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

CHECKPOINT = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=1
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Checkpoint:", CHECKPOINT)
print("Device:", device)
print("Regression outputs:", model.config.num_labels)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Checkpoint: xlm-roberta-base
Device: cpu
Regression outputs: 1


In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [7]:
import os

for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in ["JUDGEBERT", ".cache"]]
    for file in files:
        if file.lower().endswith((".csv", ".xlsx", ".json", ".jsonl")):
            print(os.path.join(root, file))

/content/lmp_final - lmp_semantic_review(1).csv
/content/.config/.last_update_check.json
/content/sample_data/anscombe.json
/content/sample_data/california_housing_train.csv
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_test.csv


In [8]:
import pandas as pd

df = pd.read_csv(
    "/content/lmp_final - lmp_semantic_review(1).csv"
)

if "LMP score" in df.columns and "LMP" not in df.columns:
    df = df.rename(columns={"LMP score": "LMP"})

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (357, 8)
Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP', 'Human_Assessment']


,ID,Original,Language,Domain,Source,Simplified,LMP,Human_Assessment
0,en_tos_001,"As a result, these Terms of Service help defin...",en,tos_privacy,google_tos,"Because of this, these Terms of Service help d...",10,Legal meaning and key details preserved — Key ...
1,en_tos_002,"Our services are designed to work together, ma...",en,tos_privacy,google_tos,Our services are designed to work together so ...,9,Legal meaning and key details preserved — Key ...
2,en_tos_003,"We also make various policies, help centers, a...",en,tos_privacy,google_tos,"We also provide various policies, help centers...",9,Legal meaning and key details preserved — Key ...
3,en_tos_004,"comply with applicable laws, including export ...",en,tos_privacy,google_tos,"Comply with applicable laws, including export ...",10,Legal meaning and key details preserved — Key ...
4,en_tos_005,using our services (including the content they...,en,tos_privacy,google_tos,Do not use our services (including the content...,10,Legal meaning and key details preserved — Key ...


In [9]:
import numpy as np
from sklearn.model_selection import train_test_split

model_df = df[
    ["ID", "Original", "Simplified", "Language", "Domain", "LMP"]
].copy()

model_df["LMP"] = pd.to_numeric(model_df["LMP"], errors="coerce")
model_df = model_df.dropna(
    subset=["Original", "Simplified", "Language", "Domain", "LMP"]
).reset_index(drop=True)

model_df["LMP"] = model_df["LMP"].astype(float)

train_idx, test_idx = train_test_split(
    np.arange(len(model_df)),
    test_size=0.20,
    random_state=42
)

train_df = model_df.iloc[train_idx].reset_index(drop=True)
test_df = model_df.iloc[test_idx].reset_index(drop=True)

train_part, val_part = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42
)

train_part = train_part.reset_index(drop=True)
val_part = val_part.reset_index(drop=True)

print("Total:", len(model_df))
print("Training:", len(train_part))
print("Validation:", len(val_part))
print("Test:", len(test_df))

Total: 357
Training: 242
Validation: 43
Test: 72


In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CHECKPOINT = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=1
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Device:", device)
print("Model ready for LMP regression.")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Device: cuda
Model ready for LMP regression.


In [11]:
!pip -q install datasets

In [12]:
from datasets import Dataset

MAX_LENGTH = 256

def tokenize_pairs(batch):
    return tokenizer(
        batch["Original"],
        batch["Simplified"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_data = Dataset.from_pandas(
    train_part[["Original", "Simplified", "LMP"]]
)
val_data = Dataset.from_pandas(
    val_part[["Original", "Simplified", "LMP"]]
)
test_data = Dataset.from_pandas(
    test_df[["Original", "Simplified", "LMP"]]
)

train_data = train_data.map(tokenize_pairs, batched=True)
val_data = val_data.map(tokenize_pairs, batched=True)
test_data = test_data.map(tokenize_pairs, batched=True)

train_data = train_data.rename_column("LMP", "labels")
val_data = val_data.rename_column("LMP", "labels")
test_data = test_data.rename_column("LMP", "labels")

print("Training:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))
print("Columns:", train_data.column_names)

Map:   0%|          | 0/242 [00:00<?, ? examples/s]

Map:   0%|          | 0/43 [00:00<?, ? examples/s]

Map:   0%|          | 0/72 [00:00<?, ? examples/s]

Training: 242
Validation: 43
Test: 72
Columns: ['Original', 'Simplified', 'labels', 'input_ids', 'attention_mask']


In [13]:
!pip -q install accelerate scikit-learn scipy

In [14]:
import numpy as np
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.asarray(predictions).reshape(-1)
    labels = np.asarray(labels).reshape(-1)

    pearson = pearsonr(labels, predictions)[0] if len(labels) > 1 else 0.0
    spearman = spearmanr(labels, predictions)[0] if len(labels) > 1 else 0.0

    return {
        "rmse": float(np.sqrt(mean_squared_error(labels, predictions))),
        "mae": float(mean_absolute_error(labels, predictions)),
        "pearson": float(pearson),
        "spearman": float(spearman),
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="/content/judgebert_lmp_results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_rmse",
    greater_is_better=False,
    fp16=True,
    logging_strategy="epoch",
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Training configuration ready.")
print("GPU:", torch.cuda.get_device_name(0))
print("Epochs:", training_args.num_train_epochs)

Training configuration ready.
GPU: Tesla T4
Epochs: 5


In [15]:
train_result = trainer.train()

print("\nTraining completed!")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation RMSE:", trainer.state.best_metric)

Epoch,Training Loss,Validation Loss,Rmse,Mae,Pearson,Spearman
1,61.921245,26.063982,5.105290,4.921239,0.113955,0.136967
2,20.119991,6.526847,2.554769,2.321039,-0.347197,-0.360353
3,5.774424,1.994349,1.412214,1.194222,-0.002032,0.059111
4,2.039555,1.588119,1.260206,0.928688,0.565164,0.333494
5,1.690267,1.505611,1.227034,0.894713,0.569618,0.358591


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed!
Best checkpoint: /content/judgebert_lmp_results/checkpoint-155
Best validation RMSE: 1.2270335368148846


In [16]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

# Evaluate once on the held-out test set
test_output = trainer.predict(test_data)

y_true = np.asarray(test_output.label_ids).reshape(-1)
y_pred = np.asarray(test_output.predictions).reshape(-1)

# LMP scores are on a 1–10 scale; clip predictions to that range
y_pred = np.clip(y_pred, 1, 10)

test_results = {
    "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
    "MAE": mean_absolute_error(y_true, y_pred),
    "Pearson": pearsonr(y_true, y_pred).statistic,
    "Spearman": spearmanr(y_true, y_pred).statistic,
}

print("JUDGEBERT-inspired XLM-R test results")
for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

JUDGEBERT-inspired XLM-R test results
RMSE: 1.2326
MAE: 0.9753
Pearson: 0.5780
Spearman: 0.4165


In [17]:
SAVE_PATH = "/content/judgebert_lmp_final"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved to:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/judgebert_lmp_final


In [18]:
import numpy as np
import pandas as pd
import torch

# Use the best model selected during training
trainer.model.eval()

all_data = Dataset.from_pandas(
    model_df[["Original", "Simplified", "LMP", "Language", "Domain"]]
)

def tokenize_all(batch):
    return tokenizer(
        batch["Original"],
        batch["Simplified"],
        truncation=True,
        max_length=256,
    )

all_data = all_data.map(tokenize_all, batched=True)
all_data = all_data.rename_column("LMP", "labels")

all_predictions = trainer.predict(all_data)
predicted_scores = np.asarray(
    all_predictions.predictions
).reshape(-1)

# Keep predictions within the project's 1–10 score scale
predicted_scores = np.clip(predicted_scores, 1, 10)

results_df = model_df.copy()
results_df["Predicted_LMP"] = np.round(predicted_scores, 2)

# Provisional threshold; document and justify it in the paper
THRESHOLD = 8.0

results_df["Preservation_Decision"] = np.where(
    results_df["Predicted_LMP"] >= THRESHOLD,
    "Preserved",
    "Review: possible meaning loss",
)

display(results_df.head(10))
print("Total predictions:", len(results_df))

Map:   0%|          | 0/357 [00:00<?, ? examples/s]

,ID,Original,Simplified,Language,Domain,LMP,Predicted_LMP,Preservation_Decision
0,en_tos_001,"As a result, these Terms of Service help defin...","Because of this, these Terms of Service help d...",en,tos_privacy,10.0,8.91,Preserved
1,en_tos_002,"Our services are designed to work together, ma...",Our services are designed to work together so ...,en,tos_privacy,9.0,9.05,Preserved
2,en_tos_003,"We also make various policies, help centers, a...","We also provide various policies, help centers...",en,tos_privacy,9.0,8.92,Preserved
3,en_tos_004,"comply with applicable laws, including export ...","Comply with applicable laws, including export ...",en,tos_privacy,10.0,8.96,Preserved
4,en_tos_005,using our services (including the content they...,Do not use our services (including the content...,en,tos_privacy,10.0,8.99,Preserved
5,en_tos_006,We need your permission if your intellectual p...,We need your permission if your intellectual p...,en,tos_privacy,10.0,9.02,Preserved
6,en_tos_007,This license lasts for as long as your content...,This license lasts as long as your content is ...,en,tos_privacy,10.0,8.98,Preserved
7,en_tos_008,Google services may frequently use the Interne...,Google services may use the Internet frequentl...,en,tos_privacy,9.0,8.88,Preserved
8,en_tos_009,Some of our services include software that’s o...,Some of our services include software offered ...,en,tos_privacy,9.0,8.90,Preserved
9,en_tos_010,"The information Google collects, and how that ...",The information Google collects and how it is ...,en,tos_privacy,9.0,8.87,Preserved


Total predictions: 357


In [19]:
OUTPUT_PATH = "/content/lmp_prediction_results.csv"

results_df.to_csv(OUTPUT_PATH, index=False)

print("Results exported:", OUTPUT_PATH)

Results exported: /content/lmp_prediction_results.csv


In [20]:
print("Prediction distribution:")
print(results_df["Preservation_Decision"].value_counts())

print("\nPredicted LMP statistics:")
print(results_df["Predicted_LMP"].describe())

print("\nLanguage-wise results:")
display(
    results_df.groupby("Language")["Predicted_LMP"]
    .agg(["count", "mean", "std"])
    .round(2)
)

print("\nDomain-wise results:")
display(
    results_df.groupby("Domain")["Predicted_LMP"]
    .agg(["count", "mean", "std"])
    .round(2)
)

Prediction distribution:
Preservation_Decision
Preserved                        344
Review: possible meaning loss     13
Name: count, dtype: int64

Predicted LMP statistics:
count    357.000000
mean       8.863221
std        0.329144
min        7.340000
25%        8.780000
50%        8.980000
75%        9.100000
max        9.150000
Name: Predicted_LMP, dtype: float64

Language-wise results:


,count,mean,std
Language,,,
en,205,8.70,0.35
hi,152,9.09,0.06



Domain-wise results:


,count,mean,std
Domain,,,
rental,130,8.77,0.45
tos_privacy,227,8.92,0.21


In [21]:
difficult_cases = results_df.copy()

difficult_cases["Score_Difference"] = (
    difficult_cases["Predicted_LMP"] - difficult_cases["LMP"]
).abs()

difficult_cases = difficult_cases.sort_values(
    "Score_Difference", ascending=False
)

display(
    difficult_cases[
        [
            "Original",
            "Simplified",
            "Language",
            "Domain",
            "LMP",
            "Predicted_LMP",
            "Score_Difference",
        ]
    ].head(20)
)

,Original,Simplified,Language,Domain,LMP,Predicted_LMP,Score_Difference
205,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेव...",उपयोग करते समय सहमत होने वाले नियमों का वर्णन ...,hi,tos_privacy,5.0,9.10,4.10
266,हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव क...,"हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धो...",hi,tos_privacy,5.0,8.99,3.99
191,The Landlord may rel et the Premises or any pa...,The Landlord may sublet the Premises or part o...,en,rental,4.0,7.93,3.93
202,Within 30 days of the termination of this leas...,Within 30 days after the lease ends and Tenant...,en,rental,4.0,7.61,3.61
180,It is the intent of this Lease and agreed to b...,It is agreed that the Tenant will pay the Base...,en,rental,4.0,7.60,3.60
183,Should the Tenant remain in possession of the ...,If the Tenant stays in the Premises with the L...,en,rental,4.0,7.34,3.34
280,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाण...,PhonePe आपसे विशिष्ट जानकारी का अनुरोध कर सकता...,hi,tos_privacy,6.0,9.13,3.13
265,हम व्यक्तिगत जानकारी को भारत के भीतर संग्रहीत ...,हम भारत में व्यक्तिगत जानकारी को संग्रहीत करते...,hi,tos_privacy,6.0,9.12,3.12
116,To use less information that's connected to in...,"To use less personal information, we sometimes...",en,tos_privacy,5.0,8.11,3.11
332,अचल संपति को किराये पर देने की बाबत किसी प्रका...,अचल संपत्ति को किराये पर देने की कोई रूकावट नह...,hi,rental,6.0,9.09,3.09


In [22]:
difficult_cases = results_df.copy()

difficult_cases["Score_Difference"] = (
    difficult_cases["Predicted_LMP"] - difficult_cases["LMP"]
).abs()

difficult_cases = difficult_cases.sort_values(
    "Score_Difference",
    ascending=False
)

display(
    difficult_cases[
        [
            "ID",
            "Original",
            "Simplified",
            "Language",
            "Domain",
            "LMP",
            "Predicted_LMP",
            "Score_Difference",
        ]
    ].head(20)
)

,ID,Original,Simplified,Language,Domain,LMP,Predicted_LMP,Score_Difference
205,hi_tos_001,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेव...",उपयोग करते समय सहमत होने वाले नियमों का वर्णन ...,hi,tos_privacy,5.0,9.10,4.10
266,hi_tos_062,हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव क...,"हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धो...",hi,tos_privacy,5.0,8.99,3.99
191,en_rental_046,The Landlord may rel et the Premises or any pa...,The Landlord may sublet the Premises or part o...,en,rental,4.0,7.93,3.93
202,en_rental_057,Within 30 days of the termination of this leas...,Within 30 days after the lease ends and Tenant...,en,rental,4.0,7.61,3.61
180,en_rental_035,It is the intent of this Lease and agreed to b...,It is agreed that the Tenant will pay the Base...,en,rental,4.0,7.60,3.60
183,en_rental_038,Should the Tenant remain in possession of the ...,If the Tenant stays in the Premises with the L...,en,rental,4.0,7.34,3.34
280,hi_tos_076,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाण...,PhonePe आपसे विशिष्ट जानकारी का अनुरोध कर सकता...,hi,tos_privacy,6.0,9.13,3.13
265,hi_tos_061,हम व्यक्तिगत जानकारी को भारत के भीतर संग्रहीत ...,हम भारत में व्यक्तिगत जानकारी को संग्रहीत करते...,hi,tos_privacy,6.0,9.12,3.12
116,en_tos_117,To use less information that's connected to in...,"To use less personal information, we sometimes...",en,tos_privacy,5.0,8.11,3.11
332,hi_rental_047,अचल संपति को किराये पर देने की बाबत किसी प्रका...,अचल संपत्ति को किराये पर देने की कोई रूकावट नह...,hi,rental,6.0,9.09,3.09


In [23]:
results_df["Prediction_Error"] = (
    results_df["Predicted_LMP"] - results_df["LMP"]
)

results_df["Absolute_Error"] = results_df["Prediction_Error"].abs()

print("Mean absolute error:", round(results_df["Absolute_Error"].mean(), 4))

print("\nLargest overestimations:")
display(
    results_df.sort_values("Prediction_Error", ascending=False)[
        ["ID", "Language", "Domain", "LMP",
         "Predicted_LMP", "Prediction_Error",
         "Original", "Simplified"]
    ].head(10)
)

print("\nLargest underestimations:")
display(
    results_df.sort_values("Prediction_Error")[
        ["ID", "Language", "Domain", "LMP",
         "Predicted_LMP", "Prediction_Error",
         "Original", "Simplified"]
    ].head(10)
)

Mean absolute error: 0.9084

Largest overestimations:


,ID,Language,Domain,LMP,Predicted_LMP,Prediction_Error,Original,Simplified
205,hi_tos_001,hi,tos_privacy,5.0,9.10,4.10,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेव...",उपयोग करते समय सहमत होने वाले नियमों का वर्णन ...
266,hi_tos_062,hi,tos_privacy,5.0,8.99,3.99,हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव क...,"हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धो..."
191,en_rental_046,en,rental,4.0,7.93,3.93,The Landlord may rel et the Premises or any pa...,The Landlord may sublet the Premises or part o...
202,en_rental_057,en,rental,4.0,7.61,3.61,Within 30 days of the termination of this leas...,Within 30 days after the lease ends and Tenant...
180,en_rental_035,en,rental,4.0,7.60,3.60,It is the intent of this Lease and agreed to b...,It is agreed that the Tenant will pay the Base...
183,en_rental_038,en,rental,4.0,7.34,3.34,Should the Tenant remain in possession of the ...,If the Tenant stays in the Premises with the L...
280,hi_tos_076,hi,tos_privacy,6.0,9.13,3.13,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाण...,PhonePe आपसे विशिष्ट जानकारी का अनुरोध कर सकता...
265,hi_tos_061,hi,tos_privacy,6.0,9.12,3.12,हम व्यक्तिगत जानकारी को भारत के भीतर संग्रहीत ...,हम भारत में व्यक्तिगत जानकारी को संग्रहीत करते...
116,en_tos_117,en,tos_privacy,5.0,8.11,3.11,To use less information that's connected to in...,"To use less personal information, we sometimes..."
222,hi_tos_018,hi,tos_privacy,6.0,9.09,3.09,दूसरों को या खुद को नुकसान न पहुंचाना या दूसरो...,दूसरों को या खुद को नुकसान न पहुंचाना और बुरा ...



Largest underestimations:


,ID,Language,Domain,LMP,Predicted_LMP,Prediction_Error,Original,Simplified
166,en_rental_021,en,rental,10.0,8.35,-1.65,Major maintenance and repair of the Property n...,"Major maintenance and repair of the Property, ..."
144,en_tos_147,en,tos_privacy,10.0,8.75,-1.25,In response to legal requests from third parti...,In response to legal requests from third parti...
102,en_tos_103,en,tos_privacy,10.0,8.76,-1.24,Not all content is available in spatial audio ...,Not all content is available in spatial audio ...
198,en_rental_053,en,rental,10.0,8.77,-1.23,If the Start Date is on a day other than the f...,If the start date is not on the first of the m...
168,en_rental_023,en,rental,10.0,8.80,-1.20,The Tenant will not engage in any illegal trad...,The Tenant will not engage in any illegal acti...
163,en_rental_018,en,rental,10.0,8.81,-1.19,"A consent by the Landlord to one assignment, s...","A consent by the Landlord to one assignment, s..."
94,en_tos_095,en,tos_privacy,10.0,8.83,-1.17,While we strive to provide a consistent and hi...,While we aim to provide a consistent and high-...
106,en_tos_107,en,tos_privacy,10.0,8.84,-1.16,"The Netflix software is developed by, or for, ...",The Netflix software is developed by Netflix a...
86,en_tos_087,en,tos_privacy,10.0,8.85,-1.15,Members of households with an existing or rece...,Members with an existing or recent Netflix acc...
97,en_tos_098,en,tos_privacy,10.0,8.85,-1.15,"Depending on the subscription plan, some Netfl...","Depending on your subscription plan, some Netf..."


In [24]:
import pandas as pd

# Select 15 largest disagreements for manual legal review
case_review = (
    results_df.assign(
        Absolute_Error=lambda x: (x["Predicted_LMP"] - x["LMP"]).abs()
    )
    .sort_values("Absolute_Error", ascending=False)
    .head(15)
    .copy()
)

case_review["Legal_Error_Type"] = "Needs manual review"
case_review["Legal_Impact"] = ""
case_review["Reviewer_Explanation"] = ""

case_review = case_review[
    [
        "ID",
        "Language",
        "Domain",
        "Original",
        "Simplified",
        "LMP",
        "Predicted_LMP",
        "Absolute_Error",
        "Legal_Error_Type",
        "Legal_Impact",
        "Reviewer_Explanation",
    ]
]

display(case_review)

,ID,Language,Domain,Original,Simplified,LMP,Predicted_LMP,Absolute_Error,Legal_Error_Type,Legal_Impact,Reviewer_Explanation
205,hi_tos_001,hi,tos_privacy,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेव...",उपयोग करते समय सहमत होने वाले नियमों का वर्णन ...,5.0,9.10,4.10,Needs manual review,,
266,hi_tos_062,hi,tos_privacy,हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव क...,"हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धो...",5.0,8.99,3.99,Needs manual review,,
191,en_rental_046,en,rental,The Landlord may rel et the Premises or any pa...,The Landlord may sublet the Premises or part o...,4.0,7.93,3.93,Needs manual review,,
202,en_rental_057,en,rental,Within 30 days of the termination of this leas...,Within 30 days after the lease ends and Tenant...,4.0,7.61,3.61,Needs manual review,,
180,en_rental_035,en,rental,It is the intent of this Lease and agreed to b...,It is agreed that the Tenant will pay the Base...,4.0,7.60,3.60,Needs manual review,,
183,en_rental_038,en,rental,Should the Tenant remain in possession of the ...,If the Tenant stays in the Premises with the L...,4.0,7.34,3.34,Needs manual review,,
280,hi_tos_076,hi,tos_privacy,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाण...,PhonePe आपसे विशिष्ट जानकारी का अनुरोध कर सकता...,6.0,9.13,3.13,Needs manual review,,
265,hi_tos_061,hi,tos_privacy,हम व्यक्तिगत जानकारी को भारत के भीतर संग्रहीत ...,हम भारत में व्यक्तिगत जानकारी को संग्रहीत करते...,6.0,9.12,3.12,Needs manual review,,
116,en_tos_117,en,tos_privacy,To use less information that's connected to in...,"To use less personal information, we sometimes...",5.0,8.11,3.11,Needs manual review,,
332,hi_rental_047,hi,rental,अचल संपति को किराये पर देने की बाबत किसी प्रका...,अचल संपत्ति को किराये पर देने की कोई रूकावट नह...,6.0,9.09,3.09,Needs manual review,,


In [25]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

for _, row in case_review.iterrows():
    print("=" * 100)
    print("ID:", row["ID"])
    print("Language:", row["Language"], "| Domain:", row["Domain"])
    print("Human LMP:", row["LMP"],
          "| Predicted LMP:", row["Predicted_LMP"])
    print("\nORIGINAL:\n", row["Original"])
    print("\nSIMPLIFIED:\n", row["Simplified"])
    print()

ID: hi_tos_001
Language: hi | Domain: tos_privacy
Human LMP: 5.0 | Predicted LMP: 9.100000381469727

ORIGINAL:
 उन नियमों का वर्णन करता है, जिनसे आप हमारी सेवाओं का उपयोग करते समय सहमत होते हैं.

SIMPLIFIED:
 उपयोग करते समय सहमत होने वाले नियमों का वर्णन करता है.

ID: hi_tos_062
Language: hi | Domain: tos_privacy
Human LMP: 5.0 | Predicted LMP: 8.989999771118164

ORIGINAL:
 हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव करके रख सकते हैं, अगर हमें लगता है कि यह, धोखाधड़ी या भविष्य में गलत इस्तेमाल को रोकने के लिए ज़रूरी है या किसी कानूनी/नियामक कार्यवाही को अधूरी रहने या उस बारे में कोई कानूनी और/या नियामक निर्देश मिलने की स्थिति में कानून के हिसाब से ज़रूरी हो।

SIMPLIFIED:
 हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धोखाधड़ी रोकने या कानूनी कार्यवाही को अधूरा नहीं रहने के लिए ज़रूरी हो।

ID: en_rental_046
Language: en | Domain: rental
Human LMP: 4.0 | Predicted LMP: 7.929999828338623

ORIGINAL:
 The Landlord may rel et the Premises or any part of the Premises for a term or terms which 

In [26]:
review_notes = {
    "hi_tos_001": (
        "Scope/detail omission",
        "Low to moderate",
        "The explicit reference to using the services is omitted."
    ),
    "hi_tos_062": (
        "Condition/purpose omission",
        "High",
        "Future misuse prevention and some legal or regulatory circumstances are omitted."
    ),
    "en_rental_046": (
        "Potential terminology change",
        "Needs legal review",
        "Relet and sublet may not be legally equivalent in this lease context."
    ),
    "en_rental_057": (
        "Condition/detail omission",
        "Moderate",
        "The explicit tenant-vacating condition and some itemization detail are omitted."
    ),
    "en_rental_035": (
        "No material change apparent",
        "Low",
        "The gross-rent arrangement and exception are substantially retained."
    ),
    "en_rental_038": (
        "No material change apparent",
        "Low",
        "The tenancy type, original terms, and one-month notice are retained."
    ),
    "hi_tos_076": (
        "No material change apparent",
        "Low",
        "Identity verification and authentication purposes are retained."
    ),
    "hi_tos_061": (
        "Retention-period omission",
        "High",
        "The purpose-based limit on the retention period is omitted."
    ),
    "en_tos_117": (
        "Technical-detail omission",
        "Moderate",
        "Anonymisation and the explicit non-identifiability outcome are omitted."
    ),
    "hi_rental_047": (
        "Authority/scope omission",
        "Moderate",
        "The explicit reference to departments and courts is omitted."
    ),
    "hi_tos_018": (
        "Threat/encouragement omission",
        "Moderate",
        "Threatening or encouraging harmful conduct is omitted."
    ),
    "hi_tos_029": (
        "Example/detail omission",
        "Low",
        "The example of changing content format is omitted."
    ),
    "hi_tos_034": (
        "Response-condition omission",
        "Moderate",
        "The ability to act without replying to the suggestion is omitted."
    ),
    "en_tos_144": (
        "Mostly preserved",
        "Low",
        "The legal-obligation limitation is retained, although relevant is omitted."
    ),
    "en_tos_126": (
        "No material change apparent",
        "Low",
        "The core meaning is retained."
    ),
}

for idx, row in case_review.iterrows():
    case_id = row["ID"]
    if case_id in review_notes:
        (
            case_review.at[idx, "Legal_Error_Type"],
            case_review.at[idx, "Legal_Impact"],
            case_review.at[idx, "Reviewer_Explanation"],
        ) = review_notes[case_id]

display(
    case_review[
        [
            "ID",
            "Legal_Error_Type",
            "Legal_Impact",
            "Reviewer_Explanation",
        ]
    ]
)

,ID,Legal_Error_Type,Legal_Impact,Reviewer_Explanation
205,hi_tos_001,Scope/detail omission,Low to moderate,The explicit reference to using the services is omitted.
266,hi_tos_062,Condition/purpose omission,High,Future misuse prevention and some legal or regulatory circumstances are omitted.
191,en_rental_046,Potential terminology change,Needs legal review,Relet and sublet may not be legally equivalent in this lease context.
202,en_rental_057,Condition/detail omission,Moderate,The explicit tenant-vacating condition and some itemization detail are omitted.
180,en_rental_035,No material change apparent,Low,The gross-rent arrangement and exception are substantially retained.
183,en_rental_038,No material change apparent,Low,"The tenancy type, original terms, and one-month notice are retained."
280,hi_tos_076,No material change apparent,Low,Identity verification and authentication purposes are retained.
265,hi_tos_061,Retention-period omission,High,The purpose-based limit on the retention period is omitted.
116,en_tos_117,Technical-detail omission,Moderate,Anonymisation and the explicit non-identifiability outcome are omitted.
332,hi_rental_047,Authority/scope omission,Moderate,The explicit reference to departments and courts is omitted.


In [27]:
case_review.to_csv(
    "/content/lmp_qualitative_case_analysis.csv",
    index=False
)

print("Saved: /content/lmp_qualitative_case_analysis.csv")

Saved: /content/lmp_qualitative_case_analysis.csv


In [28]:

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "/content/judgebert_lmp_final"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

sanity_pairs = [
    {
        "Test": "Identical",
        "Original": "The tenant must pay rent on time.",
        "Simplified": "The tenant must pay rent on time."
    },
    {
        "Test": "Identical",
        "Original": "We store personal information in India.",
        "Simplified": "We store personal information in India."
    },
    {
        "Test": "Unrelated",
        "Original": "The tenant must pay rent on time.",
        "Simplified": "The Earth revolves around the Sun."
    },
    {
        "Test": "Unrelated",
        "Original": "We store personal information in India.",
        "Simplified": "The recipe requires two cups of flour."
    }
]

def predict_lmp(original, simplified):
    inputs = tokenizer(
        original,
        simplified,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        prediction = model(**inputs).logits.squeeze().item()

    return round(max(1.0, min(10.0, prediction)), 2)

for pair in sanity_pairs:
    pair["Predicted_LMP"] = predict_lmp(
        pair["Original"], pair["Simplified"]
    )

sanity_df = pd.DataFrame(sanity_pairs)
display(sanity_df)

sanity_df.to_csv("/content/lmp_sanity_checks.csv", index=False)
print("Saved: /content/lmp_sanity_checks.csv")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

,Test,Original,Simplified,Predicted_LMP
0,Identical,The tenant must pay rent on time.,The tenant must pay rent on time.,8.95
1,Identical,We store personal information in India.,We store personal information in India.,8.91
2,Unrelated,The tenant must pay rent on time.,The Earth revolves around the Sun.,8.55
3,Unrelated,We store personal information in India.,The recipe requires two cups of flour.,8.72


Saved: /content/lmp_sanity_checks.csv


In [29]:

# Generalization analysis using the predictions already generated
import pandas as pd

results_df["Absolute_Error"] = (
    results_df["Predicted_LMP"] - results_df["LMP"]
).abs()

print("Performance by language")
language_results = results_df.groupby("Language").apply(
    lambda g: pd.Series({
        "N": len(g),
        "MAE": g["Absolute_Error"].mean(),
        "Mean_Human_LMP": g["LMP"].mean(),
        "Mean_Predicted_LMP": g["Predicted_LMP"].mean()
    }),
    include_groups=False
)
display(language_results.round(3))

print("\nPerformance by legal domain")
domain_results = results_df.groupby("Domain").apply(
    lambda g: pd.Series({
        "N": len(g),
        "MAE": g["Absolute_Error"].mean(),
        "Mean_Human_LMP": g["LMP"].mean(),
        "Mean_Predicted_LMP": g["Predicted_LMP"].mean()
    }),
    include_groups=False
)
display(domain_results.round(3))

language_results.to_csv("/content/generalization_by_language.csv")
domain_results.to_csv("/content/generalization_by_domain.csv")

print("\nSaved both generalization analysis files.")

Performance by language


,N,MAE,Mean_Human_LMP,Mean_Predicted_LMP
Language,,,,
en,205.0,0.862,8.478,8.695
hi,152.0,0.971,9.007,9.090



Performance by legal domain


,N,MAE,Mean_Human_LMP,Mean_Predicted_LMP
Domain,,,,
rental,130.0,0.913,8.631,8.770
tos_privacy,227.0,0.906,8.744,8.917



Saved both generalization analysis files.


In [30]:

from sklearn.model_selection import train_test_split

# Use the existing model_df containing the 357 labeled examples.
# Keep the same XLM-R checkpoint and regression objective.
domain_col = "Domain"
label_col = "LMP"

print("Domain counts:")
print(model_df[domain_col].value_counts())

print("\nAvailable columns:")
print(model_df.columns.tolist())

assert model_df[domain_col].nunique() == 2
assert model_df[label_col].notna().all()

print("\nSetup verified. Ready for held-out-domain training.")

Domain counts:
Domain
tos_privacy    227
rental         130
Name: count, dtype: int64

Available columns:
['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP']

Setup verified. Ready for held-out-domain training.


In [32]:

import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

CHECKPOINT = "xlm-roberta-base"
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 5
SEED = 42

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def metric_fn(eval_pred):
    predictions, labels = eval_pred
    predictions = np.asarray(predictions).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    predictions = np.clip(predictions, 1, 10)

    return {
        "rmse": float(np.sqrt(mean_squared_error(labels, predictions))),
        "mae": float(mean_absolute_error(labels, predictions)),
        "pearson": float(pearsonr(labels, predictions).statistic)
            if np.std(predictions) > 0 and np.std(labels) > 0 else 0.0,
        "spearman": float(spearmanr(labels, predictions).statistic)
            if np.std(predictions) > 0 and np.std(labels) > 0 else 0.0,
    }


def run_domain_experiment(train_domain, test_domain):
    print(f"\n{'=' * 65}")
    print(f"Train domain: {train_domain} | Test domain: {test_domain}")
    print("=" * 65)

    train_df = model_df[model_df["Domain"] == train_domain].copy()
    test_df = model_df[model_df["Domain"] == test_domain].copy()

    # Validation is drawn only from the training domain.
    train_part, val_part = train_test_split(
        train_df,
        test_size=0.15,
        random_state=SEED,
        stratify=train_df["Language"]
    )

    def prepare_dataset(frame):
        ds = Dataset.from_pandas(
            frame[["Original", "Simplified", "LMP"]].reset_index(drop=True)
        )

        def tokenize_batch(batch):
            return tokenizer(
                batch["Original"],
                batch["Simplified"],
                truncation=True,
                max_length=MAX_LENGTH
            )

        ds = ds.map(tokenize_batch, batched=True)
        ds = ds.rename_column("LMP", "labels")
        ds.set_format(
            type="torch",
            columns=["input_ids", "attention_mask", "labels"]
        )
        return ds

    train_data = prepare_dataset(train_part)
    val_data = prepare_dataset(val_part)
    test_data = prepare_dataset(test_df)

    # Dynamically pad each batch to the longest sequence in that batch.
    data_collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        padding=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        CHECKPOINT,
        num_labels=1
    )

    output_dir = f"/content/domain_exp_{train_domain}_to_{test_domain}"

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_rmse",
        greater_is_better=False,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        data_seed=SEED,
        report_to="none",
        save_total_limit=1
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=val_data,
        data_collator=data_collator,
        compute_metrics=metric_fn
    )

    trainer.train()

    # Evaluate only after training, on the entirely held-out domain.
    prediction_output = trainer.predict(test_data)
    predictions = np.asarray(
        prediction_output.predictions
    ).reshape(-1)
    predictions = np.clip(predictions, 1, 10)

    labels = test_df["LMP"].to_numpy(dtype=float)

    result = {
        "Train_Domain": train_domain,
        "Test_Domain": test_domain,
        "Train_N": len(train_part),
        "Validation_N": len(val_part),
        "Test_N": len(test_df),
        "RMSE": np.sqrt(mean_squared_error(labels, predictions)),
        "MAE": mean_absolute_error(labels, predictions),
        "Pearson": pearsonr(labels, predictions).statistic
            if np.std(predictions) > 0 and np.std(labels) > 0 else np.nan,
        "Spearman": spearmanr(labels, predictions).statistic
            if np.std(predictions) > 0 and np.std(labels) > 0 else np.nan,
    }

    print(pd.Series(result).to_string())

    return result


# Run both directions.
domain_results = []

domain_results.append(
    run_domain_experiment("rental", "tos_privacy")
)

domain_results.append(
    run_domain_experiment("tos_privacy", "rental")
)

domain_results_df = pd.DataFrame(domain_results)

print("\nFINAL HELD-OUT-DOMAIN RESULTS")
display(domain_results_df.round(4))

domain_results_df.to_csv(
    "/content/held_out_domain_results.csv",
    index=False
)

print("Saved: /content/held_out_domain_results.csv")


Train domain: rental | Test domain: tos_privacy


Map:   0%|          | 0/110 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Train_Domain    tos_privacy
Test_Domain          rental
Train_N                 192
Validation_N             35
Test_N                  130
RMSE               1.761515
MAE                1.539754
Pearson            0.309955
Spearman           0.301214

FINAL HELD-OUT-DOMAIN RESULTS


,Train_Domain,Test_Domain,Train_N,Validation_N,Test_N,RMSE,MAE,Pearson,Spearman
0,rental,tos_privacy,110,20,227,3.6670,3.4596,0.0281,-0.0017
1,tos_privacy,rental,192,35,130,1.7615,1.5398,0.3100,0.3012


Saved: /content/held_out_domain_results.csv


In [33]:

import pandas as pd
import numpy as np

df_audit = model_df.copy()

# 1. Dataset size and missing values
print("=== DATASET OVERVIEW ===")
print("Total pairs:", len(df_audit))
print("\nMissing values:")
print(df_audit[["Original", "Simplified", "Language", "Domain", "LMP"]]
      .isna().sum())

# 2. Duplicate pair check (case- and whitespace-insensitive)
df_audit["pair_key"] = (
    df_audit["Original"].fillna("").str.strip().str.lower()
    + " || "
    + df_audit["Simplified"].fillna("").str.strip().str.lower()
)

print("\n=== DUPLICATE CHECK ===")
print("Rows belonging to repeated sentence pairs:",
      int(df_audit["pair_key"].duplicated(keep=False).sum()))
print("Repeated pair occurrences beyond the first:",
      int(df_audit["pair_key"].duplicated().sum()))

# Check whether exact pairs have conflicting LMP labels
pair_label_counts = df_audit.groupby("pair_key")["LMP"].nunique()
print("Repeated pairs with conflicting LMP labels:",
      int((pair_label_counts[pair_label_counts > 1]).shape[0]))

# 3. Human LMP score distribution
print("\n=== LMP SCORE DISTRIBUTION ===")
display(
    df_audit["LMP"].value_counts().sort_index()
    .rename_axis("LMP")
    .to_frame("Count")
)

# 4. Score distribution by language
print("\n=== SCORES BY LANGUAGE ===")
display(
    df_audit.groupby("Language")["LMP"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(3)
)

# 5. Score distribution by legal domain
print("\n=== SCORES BY DOMAIN ===")
display(
    df_audit.groupby("Domain")["LMP"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(3)
)

# 6. Language x domain counts
print("\n=== LANGUAGE / DOMAIN COUNTS ===")
display(pd.crosstab(df_audit["Language"], df_audit["Domain"]))

# 7. Identical original/simplified pairs in the dataset
identical_mask = (
    df_audit["Original"].fillna("").str.strip().str.lower()
    == df_audit["Simplified"].fillna("").str.strip().str.lower()
)
print("\nIdentical original/simplified pairs:", int(identical_mask.sum()))

print("\nAudit complete. No dataset values were modified.")

=== DATASET OVERVIEW ===
Total pairs: 357

Missing values:
Original      0
Simplified    0
Language      0
Domain        0
LMP           0
dtype: int64

=== DUPLICATE CHECK ===
Rows belonging to repeated sentence pairs: 0
Repeated pair occurrences beyond the first: 0
Repeated pairs with conflicting LMP labels: 0

=== LMP SCORE DISTRIBUTION ===


,Count
LMP,
4.0,4
5.0,5
6.0,17
7.0,34
8.0,71
9.0,102
10.0,124



=== SCORES BY LANGUAGE ===


,count,mean,std,min,max
Language,,,,,
en,205,8.478,1.352,4.0,10.0
hi,152,9.007,1.247,5.0,10.0



=== SCORES BY DOMAIN ===


,count,mean,std,min,max
Domain,,,,,
rental,130,8.631,1.490,4.0,10.0
tos_privacy,227,8.744,1.236,5.0,10.0



=== LANGUAGE / DOMAIN COUNTS ===


Domain,rental,tos_privacy
Language,,
en,59,146
hi,71,81



Identical original/simplified pairs: 14

Audit complete. No dataset values were modified.


In [34]:

from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Reconstruct the original 80/20 split.
all_idx = np.arange(len(model_df))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42
)

# Reconstruct the training/validation split.
train_part_idx, val_idx = train_test_split(
    train_idx, test_size=0.15, random_state=42
)

train_df = model_df.iloc[train_part_idx].copy()
val_df = model_df.iloc[val_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("=== SPLIT SIZES ===")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\n=== LMP DISTRIBUTION BY SPLIT ===")
for split_name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"\n{split_name}:")
    print("Mean LMP:", round(split_df["LMP"].mean(), 3))
    print("Std LMP:", round(split_df["LMP"].std(), 3))
    print("Score counts:")
    print(split_df["LMP"].value_counts().sort_index().to_dict())

print("\n=== LANGUAGE / DOMAIN DISTRIBUTION ===")
for split_name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"\n{split_name}:")
    display(pd.crosstab(split_df["Language"], split_df["Domain"]))

# Check for identical normalized pairs crossing split boundaries.
def normalize_pair(frame):
    return (
        frame["Original"].fillna("").str.strip().str.lower()
        + " || "
        + frame["Simplified"].fillna("").str.strip().str.lower()
    )

train_pairs = set(normalize_pair(train_df))
val_pairs = set(normalize_pair(val_df))
test_pairs = set(normalize_pair(test_df))

print("\n=== CROSS-SPLIT IDENTICAL PAIRS ===")
print("Train ∩ Validation:", len(train_pairs & val_pairs))
print("Train ∩ Test:", len(train_pairs & test_pairs))
print("Validation ∩ Test:", len(val_pairs & test_pairs))

# Save the split IDs for reproducibility.
split_ids = pd.concat([
    pd.DataFrame({"row_index": train_part_idx, "split": "train"}),
    pd.DataFrame({"row_index": val_idx, "split": "validation"}),
    pd.DataFrame({"row_index": test_idx, "split": "test"}),
], ignore_index=True)

split_ids.to_csv("/content/lmp_split_audit.csv", index=False)
print("\nSaved /content/lmp_split_audit.csv")

=== SPLIT SIZES ===
Training: 242
Validation: 43
Test: 72

=== LMP DISTRIBUTION BY SPLIT ===

Train:
Mean LMP: 8.723
Std LMP: 1.321
Score counts:
{4.0: 3, 5.0: 4, 6.0: 8, 7.0: 22, 8.0: 54, 9.0: 65, 10.0: 86}

Validation:
Mean LMP: 8.744
Std LMP: 1.382
Score counts:
{4.0: 1, 6.0: 2, 7.0: 5, 8.0: 5, 9.0: 15, 10.0: 15}

Test:
Mean LMP: 8.611
Std LMP: 1.359
Score counts:
{5.0: 1, 6.0: 7, 7.0: 7, 8.0: 12, 9.0: 22, 10.0: 23}

=== LANGUAGE / DOMAIN DISTRIBUTION ===

Train:


Domain,rental,tos_privacy
Language,,
en,47,94
hi,48,53



Validation:


Domain,rental,tos_privacy
Language,,
en,5,18
hi,8,12



Test:


Domain,rental,tos_privacy
Language,,
en,7,34
hi,15,16



=== CROSS-SPLIT IDENTICAL PAIRS ===
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0

Saved /content/lmp_split_audit.csv


In [35]:

# Inspect whether TF-IDF was fitted before or after splitting.
# Run this in the same notebook where you trained the baselines.

print("Relevant variables currently in memory:")
for name in [
    "tfidf", "tfidf_vectorizer", "vectorizer",
    "X_train_tfidf", "X_test_tfidf",
    "tfidf_model", "lr_tfidf", "tfidf_regressor"
]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}")

print("\nSearch your notebook cells for these patterns:")
print("- TfidfVectorizer(")
print("- fit_transform(")
print("- train_test_split(")
print("- LinearRegression(")

Relevant variables currently in memory:

Search your notebook cells for these patterns:
- TfidfVectorizer(
- fit_transform(
- train_test_split(
- LinearRegression(


In [37]:

# Check saved prediction files for IDs and compare their test rows.
import os
import pandas as pd

for path in [
    "/content/tfidf_clean_test_predictions.csv",
    "/content/lmp_prediction_results.csv",
    "/content/held_out_domain_results.csv"
]:
    print(f"\n{path}:")
    if os.path.exists(path):
        temp = pd.read_csv(path)
        print("Shape:", temp.shape)
        print("Columns:", temp.columns.tolist())
        if "ID" in temp.columns:
            print("First IDs:", temp["ID"].head(10).tolist())
    else:
        print("Not found")


/content/tfidf_clean_test_predictions.csv:
Shape: (72, 8)
Columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP', 'TFIDF_Cosine', 'Predicted_LMP']
First IDs: ['hi_tos_018', 'en_tos_043', 'hi_rental_006', 'hi_rental_015', 'en_tos_057', 'hi_rental_050', 'en_tos_016', 'en_tos_058', 'hi_tos_041', 'en_tos_125']

/content/lmp_prediction_results.csv:
Shape: (357, 8)
Columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP', 'Predicted_LMP', 'Preservation_Decision']
First IDs: ['en_tos_001', 'en_tos_002', 'en_tos_003', 'en_tos_004', 'en_tos_005', 'en_tos_006', 'en_tos_007', 'en_tos_008', 'en_tos_009', 'en_tos_010']

/content/held_out_domain_results.csv:
Shape: (2, 9)
Columns: ['Train_Domain', 'Test_Domain', 'Train_N', 'Validation_N', 'Test_N', 'RMSE', 'MAE', 'Pearson', 'Spearman']


In [38]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Reconstruct the split from the original dataset
audit_df = model_df.reset_index(drop=True).copy()

all_idx = np.arange(len(audit_df))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42
)

expected_test = audit_df.iloc[test_idx].copy()

tfidf_test = pd.read_csv("/content/tfidf_clean_test_predictions.csv")
xlmr_all = pd.read_csv("/content/lmp_prediction_results.csv")

print("Expected test rows:", len(expected_test))
print("TF-IDF test rows:", len(tfidf_test))

# Check whether the TF-IDF IDs match the reconstructed test split
expected_ids = set(expected_test["ID"].astype(str))
tfidf_ids = set(tfidf_test["ID"].astype(str))

print("TF-IDF IDs match expected test IDs:", tfidf_ids == expected_ids)
print("Missing TF-IDF IDs:", sorted(expected_ids - tfidf_ids)[:10])
print("Unexpected TF-IDF IDs:", sorted(tfidf_ids - expected_ids)[:10])

# Extract the XLM-R predictions for the same test IDs
xlmr_test = xlmr_all[xlmr_all["ID"].astype(str).isin(expected_ids)].copy()

print("\nXLM-R rows matching test IDs:", len(xlmr_test))
print("XLM-R IDs match expected test IDs:",
      set(xlmr_test["ID"].astype(str)) == expected_ids)

# Save a test-only XLM-R file for subsequent comparisons
xlmr_test.to_csv("/content/xlmr_test_predictions_audit.csv", index=False)
print("Saved /content/xlmr_test_predictions_audit.csv")

Expected test rows: 72
TF-IDF test rows: 72
TF-IDF IDs match expected test IDs: True
Missing TF-IDF IDs: []
Unexpected TF-IDF IDs: []

XLM-R rows matching test IDs: 72
XLM-R IDs match expected test IDs: True
Saved /content/xlmr_test_predictions_audit.csv


In [39]:

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

test_ids = set(
    pd.read_csv("/content/tfidf_clean_test_predictions.csv")["ID"].astype(str)
)

xlmr_all = pd.read_csv("/content/lmp_prediction_results.csv")
xlmr_test = xlmr_all[xlmr_all["ID"].astype(str).isin(test_ids)].copy()

assert len(xlmr_test) == 72
assert xlmr_test["ID"].astype(str).nunique() == 72

y_true = pd.to_numeric(xlmr_test["LMP"]).to_numpy()
y_pred = pd.to_numeric(xlmr_test["Predicted_LMP"]).clip(1, 10).to_numpy()

print("XLM-R — SAME 72 TEST IDs")
print(f"Pearson: {pearsonr(y_pred, y_true).statistic:.4f}")
print(f"Spearman: {spearmanr(y_pred, y_true).statistic:.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_true, y_pred):.4f}")

XLM-R — SAME 72 TEST IDs
Pearson: 0.5784
Spearman: 0.4195
RMSE: 1.2325
MAE: 0.9754


In [40]:

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr, spearmanr
import numpy as np
import pandas as pd

# Use the same dataset and row order as the TF-IDF baseline
baseline_df = model_df.reset_index(drop=True).copy()

all_idx = np.arange(len(baseline_df))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42
)
train_part_idx, val_idx = train_test_split(
    train_idx, test_size=0.15, random_state=42
)

train_df = baseline_df.iloc[train_part_idx].copy()
test_df = baseline_df.iloc[test_idx].copy()

# Pretrained multilingual sentence-embedding model
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def get_cosines(frame):
    original = model.encode(
        frame["Original"].astype(str).tolist(),
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    simplified = model.encode(
        frame["Simplified"].astype(str).tolist(),
        batch_size=16,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    return np.sum(original * simplified, axis=1)

train_cosine = get_cosines(train_df)
test_cosine = get_cosines(test_df)

y_train = train_df["LMP"].astype(float).to_numpy()
y_test = test_df["LMP"].astype(float).to_numpy()

# Fit calibration only on training data
calibrator = LinearRegression()
calibrator.fit(train_cosine.reshape(-1, 1), y_train)

predictions = np.clip(
    calibrator.predict(test_cosine.reshape(-1, 1)), 1, 10
)

print("CLEAN MINILM BASELINE")
print("-" * 40)
print("Training:", len(train_df))
print("Validation (unused):", len(val_idx))
print("Test:", len(test_df))
print(f"Pearson: {pearsonr(test_cosine, y_test).statistic:.4f}")
print(f"Spearman: {spearmanr(test_cosine, y_test).statistic:.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, predictions):.4f}")

minilm_results = test_df[
    ["ID", "Original", "Simplified", "Language", "Domain", "LMP"]
].copy()
minilm_results["Semantic_Cosine"] = test_cosine
minilm_results["Predicted_LMP"] = predictions

minilm_results.to_csv(
    "/content/minilm_clean_test_predictions.csv", index=False
)
print("Saved /content/minilm_clean_test_predictions.csv")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

CLEAN MINILM BASELINE
----------------------------------------
Training: 242
Validation (unused): 43
Test: 72
Pearson: 0.5875
Spearman: 0.6574
RMSE: 1.1058
MAE: 0.8477
Saved /content/minilm_clean_test_predictions.csv


In [41]:

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr

tfidf = pd.read_csv("/content/tfidf_clean_test_predictions.csv")
minilm = pd.read_csv("/content/minilm_clean_test_predictions.csv")
xlmr = pd.read_csv("/content/xlmr_test_predictions_audit.csv")

# Verify identical test examples
id_sets = [
    set(df_["ID"].astype(str)) for df_ in [tfidf, minilm, xlmr]
]
assert id_sets[0] == id_sets[1] == id_sets[2], \
    "Test IDs differ; investigate before comparing models."

# Align rows by ID and check labels agree
tfidf = tfidf.sort_values("ID").reset_index(drop=True)
minilm = minilm.sort_values("ID").reset_index(drop=True)
xlmr = xlmr.sort_values("ID").reset_index(drop=True)

assert tfidf["ID"].equals(minilm["ID"])
assert tfidf["ID"].equals(xlmr["ID"])
assert np.allclose(tfidf["LMP"], minilm["LMP"])
assert np.allclose(tfidf["LMP"], xlmr["LMP"])

y = tfidf["LMP"].astype(float).to_numpy()

predictions = {
    "TF-IDF": tfidf["Predicted_LMP"].astype(float).to_numpy(),
    "MiniLM": minilm["Predicted_LMP"].astype(float).to_numpy(),
    "XLM-R": xlmr["Predicted_LMP"].astype(float).to_numpy()
}

rows = []
for name, pred in predictions.items():
    rows.append({
        "Model": name,
        "Pearson": pearsonr(pred, y).statistic,
        "Spearman": spearmanr(pred, y).statistic,
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "MAE": mean_absolute_error(y, pred)
    })

comparison = pd.DataFrame(rows).round(4)
display(comparison)

comparison.to_csv("/content/lmp_model_comparison_clean.csv", index=False)
print("Saved /content/lmp_model_comparison_clean.csv")
print("All models evaluated on the same 72 test IDs.")

,Model,Pearson,Spearman,RMSE,MAE
0,TF-IDF,0.5976,0.7100,1.1634,0.7924
1,MiniLM,0.5875,0.6574,1.1058,0.8477
2,XLM-R,0.5784,0.4195,1.2325,0.9754


Saved /content/lmp_model_comparison_clean.csv
All models evaluated on the same 72 test IDs.


In [42]:

# First, inspect whether the TF-IDF objects are still available.
for name in ["vectorizer", "calibrator", "model", "tokenizer", "trainer"]:
    print(f"{name}: {'available' if name in globals() else 'not in memory'}")

vectorizer: available
calibrator: available
model: available
tokenizer: available
trainer: available


In [43]:

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# MiniLM sentence-embedding model
minilm_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Test cases
sanity_pairs = [
    {
        "Case": "Identical",
        "Original": "The tenant must pay rent on time.",
        "Simplified": "The tenant must pay rent on time."
    },
    {
        "Case": "Identical",
        "Original": "We store personal information in India.",
        "Simplified": "We store personal information in India."
    },
    {
        "Case": "Unrelated",
        "Original": "The tenant must pay rent on time.",
        "Simplified": "The Earth revolves around the Sun."
    },
    {
        "Case": "Unrelated",
        "Original": "We store personal information in India.",
        "Simplified": "The recipe requires two cups of flour."
    }
]

sanity_df = pd.DataFrame(sanity_pairs)

# 1. TF-IDF cosine similarity and calibrated LMP prediction
texts = (
    sanity_df["Original"].astype(str).tolist()
    + sanity_df["Simplified"].astype(str).tolist()
)
vectors = vectorizer.transform(texts)
n = len(sanity_df)

tfidf_cosine = np.asarray(
    vectors[:n].multiply(vectors[n:]).sum(axis=1)
).ravel()

# Existing calibrator was fitted to cosine similarity
tfidf_pred = np.clip(
    calibrator.predict(tfidf_cosine.reshape(-1, 1)), 1, 10
)

# 2. MiniLM cosine similarity and calibrated prediction
orig_emb = minilm_model.encode(
    sanity_df["Original"].tolist(), normalize_embeddings=True
)
simp_emb = minilm_model.encode(
    sanity_df["Simplified"].tolist(), normalize_embeddings=True
)
minilm_cosine = np.sum(orig_emb * simp_emb, axis=1)

# Fit a clean calibrator on the training pairs, not the test pairs
train_df = model_df.iloc[train_part_idx].copy()
orig_train = minilm_model.encode(
    train_df["Original"].astype(str).tolist(),
    normalize_embeddings=True
)
simp_train = minilm_model.encode(
    train_df["Simplified"].astype(str).tolist(),
    normalize_embeddings=True
)
train_cosine = np.sum(orig_train * simp_train, axis=1)

from sklearn.linear_model import LinearRegression
minilm_calibrator = LinearRegression().fit(
    train_cosine.reshape(-1, 1),
    train_df["LMP"].astype(float).to_numpy()
)
minilm_pred = np.clip(
    minilm_calibrator.predict(minilm_cosine.reshape(-1, 1)), 1, 10
)

# 3. Fine-tuned XLM-R regression predictions
# Uses the existing trainer and tokenizer, without retraining.
xlmr_scores = []
for _, row in sanity_df.iterrows():
    inputs = tokenizer(
        row["Original"],
        row["Simplified"],
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    device = next(trainer.model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    trainer.model.eval()
    import torch
    with torch.no_grad():
        score = trainer.model(**inputs).logits.squeeze().item()
    xlmr_scores.append(float(np.clip(score, 1, 10)))

sanity_df["TFIDF_LMP"] = np.round(tfidf_pred, 2)
sanity_df["MiniLM_Cosine"] = np.round(minilm_cosine, 4)
sanity_df["MiniLM_LMP"] = np.round(minilm_pred, 2)
sanity_df["XLMR_LMP"] = np.round(xlmr_scores, 2)

display(sanity_df)
sanity_df.to_csv("/content/lmp_sanity_check_results.csv", index=False)
print("Saved /content/lmp_sanity_check_results.csv")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,Case,Original,Simplified,TFIDF_LMP,MiniLM_Cosine,MiniLM_LMP,XLMR_LMP
0,Identical,The tenant must pay rent on time.,The tenant must pay rent on time.,9.32,1.0000,9.32,8.95
1,Identical,We store personal information in India.,We store personal information in India.,9.32,1.0000,9.32,8.91
2,Unrelated,The tenant must pay rent on time.,The Earth revolves around the Sun.,1.76,0.0338,1.00,8.55
3,Unrelated,We store personal information in India.,The recipe requires two cups of flour.,1.00,-0.1198,1.00,8.72


Saved /content/lmp_sanity_check_results.csv


In [44]:

import pandas as pd

xlmr_test = pd.read_csv("/content/xlmr_test_predictions_audit.csv")

print("Human LMP distribution:")
print(xlmr_test["LMP"].describe())

print("\nXLM-R predicted LMP distribution:")
print(xlmr_test["Predicted_LMP"].describe())

print("\nPrediction counts rounded to nearest integer:")
print(
    xlmr_test["Predicted_LMP"]
    .round()
    .value_counts()
    .sort_index()
)

print("\nLargest prediction errors:")
xlmr_test["Absolute_Error"] = (
    xlmr_test["LMP"] - xlmr_test["Predicted_LMP"]
).abs()

display(
    xlmr_test.sort_values("Absolute_Error", ascending=False)[
        ["ID", "Language", "Domain", "LMP",
         "Predicted_LMP", "Absolute_Error"]
    ].head(10)
)

Human LMP distribution:
count    72.000000
mean      8.611111
std       1.358910
min       5.000000
25%       8.000000
50%       9.000000
75%      10.000000
max      10.000000
Name: LMP, dtype: float64

XLM-R predicted LMP distribution:
count    72.00000
mean      8.88125
std       0.29847
min       7.87000
25%       8.82500
50%       8.98000
75%       9.11000
max       9.13000
Name: Predicted_LMP, dtype: float64

Prediction counts rounded to nearest integer:
Predicted_LMP
8.0     7
9.0    65
Name: count, dtype: int64

Largest prediction errors:


,ID,Language,Domain,LMP,Predicted_LMP,Absolute_Error
53,hi_tos_061,hi,tos_privacy,6.0,9.12,3.12
28,en_tos_117,en,tos_privacy,5.0,8.11,3.11
45,hi_tos_018,hi,tos_privacy,6.0,9.09,3.09
23,en_tos_091,en,tos_privacy,6.0,8.58,2.58
40,en_rental_052,en,rental,6.0,8.30,2.30
39,en_rental_050,en,rental,6.0,8.13,2.13
67,hi_rental_059,hi,rental,7.0,9.11,2.11
42,hi_tos_005,hi,tos_privacy,7.0,9.03,2.03
36,en_rental_012,en,rental,6.0,7.91,1.91
38,en_rental_040,en,rental,6.0,7.87,1.87


In [45]:

print("Training arguments:")
print("Epochs:", trainer.args.num_train_epochs)
print("Learning rate:", trainer.args.learning_rate)
print("Weight decay:", trainer.args.weight_decay)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best metric:", trainer.state.best_metric)

print("\nSplit sizes:")
print("Train:", len(train_part_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

print("\nTraining and test index overlap:")
print("Train ∩ Test:", len(set(train_part_idx) & set(test_idx)))
print("Validation ∩ Test:", len(set(val_idx) & set(test_idx)))

print("\nModel output layer:")
print(trainer.model.classifier)

Training arguments:
Epochs: 5
Learning rate: 2e-05
Weight decay: 0.01
Best checkpoint: /content/judgebert_lmp_results/checkpoint-155
Best metric: 1.2270335368148846

Split sizes:
Train: 242
Validation: 43
Test: 72

Training and test index overlap:
Train ∩ Test: 0
Validation ∩ Test: 0

Model output layer:
XLMRobertaClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=1, bias=True)
)


In [46]:

print("Model problem type:", trainer.model.config.problem_type)
print("Number of labels:", trainer.model.config.num_labels)

print("\nTrainer dataset sizes:")
print("Train:", len(trainer.train_dataset))
print("Validation:", len(trainer.eval_dataset))

print("\nTraining label dtype:")
print(trainer.train_dataset[0]["labels"].dtype)
print("Example label:", trainer.train_dataset[0]["labels"])

Model problem type: regression
Number of labels: 1

Trainer dataset sizes:
Train: 242
Validation: 43

Training label dtype:


AttributeError: 'float' object has no attribute 'dtype'

In [47]:

import torch

label = trainer.train_dataset[0]["labels"]

print("Label value:", label)
print("Label Python type:", type(label))
print("Label is numeric:", isinstance(label, (int, float)))
print("Model loss function:", trainer.model.config.problem_type)

print("\nTraining label range:")
train_labels = [float(x["labels"]) for x in trainer.train_dataset]
print("Min:", min(train_labels))
print("Max:", max(train_labels))
print("Mean:", sum(train_labels) / len(train_labels))

print("\nValidation label range:")
val_labels = [float(x["labels"]) for x in trainer.eval_dataset]
print("Min:", min(val_labels))
print("Max:", max(val_labels))
print("Mean:", sum(val_labels) / len(val_labels))

Label value: 10.0
Label Python type: <class 'float'>
Label is numeric: True
Model loss function: regression

Training label range:
Min: 4.0
Max: 10.0
Mean: 8.723140495867769

Validation label range:
Min: 4.0
Max: 10.0
Mean: 8.744186046511627


In [48]:

import torch

checks = [
    (
        "Identical",
        "The tenant must pay rent on time.",
        "The tenant must pay rent on time."
    ),
    (
        "Unrelated",
        "The tenant must pay rent on time.",
        "The Earth revolves around the Sun."
    ),
]

trainer.model.eval()

for name, original, simplified in checks:
    inputs = tokenizer(
        original,
        simplified,
        truncation=True,
        max_length=256,
        padding=True,
        return_tensors="pt"
    ).to(trainer.model.device)

    with torch.no_grad():
        score = trainer.model(**inputs).logits.squeeze().item()

    print(f"{name}: raw XLM-R score = {score:.4f}")

Identical: raw XLM-R score = 8.9531
Unrelated: raw XLM-R score = 8.5469


In [49]:

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

train_output = trainer.predict(trainer.train_dataset)
train_true = np.asarray(train_output.label_ids).reshape(-1)
train_pred = np.asarray(train_output.predictions).reshape(-1)

val_output = trainer.predict(trainer.eval_dataset)
val_true = np.asarray(val_output.label_ids).reshape(-1)
val_pred = np.asarray(val_output.predictions).reshape(-1)

def report(name, y_true, y_pred):
    print(f"\n{name}")
    print("Prediction mean:", round(float(y_pred.mean()), 4))
    print("Prediction std:", round(float(y_pred.std()), 4))
    print("MAE:", round(mean_absolute_error(y_true, y_pred), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_true, y_pred)), 4))
    print("Pearson:", round(float(pearsonr(y_true, y_pred).statistic), 4))
    print("Spearman:", round(float(spearmanr(y_true, y_pred).statistic), 4))

report("TRAIN", train_true, train_pred)
report("VALIDATION", val_true, val_pred)


TRAIN
Prediction mean: 8.8494
Prediction std: 0.3402
MAE: 0.8909
RMSE: 1.1597
Pearson: 0.5845
Spearman: 0.5116

VALIDATION
Prediction mean: 8.9122
Prediction std: 0.3113
MAE: 0.8947
RMSE: 1.227
Pearson: 0.5696
Spearman: 0.3586


In [50]:

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

# Use the labels from the existing training and validation datasets.
train_y = np.array([
    float(item["labels"]) for item in trainer.train_dataset
])
val_y = np.array([
    float(item["labels"]) for item in trainer.eval_dataset
])

# Baseline: predict the training-label mean for every validation example.
baseline_pred = np.full(len(val_y), train_y.mean())

print("Training-label mean:", round(float(train_y.mean()), 4))
print("Mean-baseline validation MAE:",
      round(mean_absolute_error(val_y, baseline_pred), 4))
print("Mean-baseline validation RMSE:",
      round(np.sqrt(mean_squared_error(val_y, baseline_pred)), 4))

print("\nXLM-R validation MAE: 0.8947")
print("XLM-R validation RMSE: 1.2270")

Training-label mean: 8.7231
Mean-baseline validation MAE: 1.0629
Mean-baseline validation RMSE: 1.3657

XLM-R validation MAE: 0.8947
XLM-R validation RMSE: 1.2270


In [51]:

from collections import Counter

train_labels = [
    int(round(float(item["labels"])))
    for item in trainer.train_dataset
]

counts = Counter(train_labels)

print("Training-label counts:")
for score in range(1, 11):
    print(f"Score {score}: {counts.get(score, 0)}")

print("\nPercentage with scores 8–10:")
high_count = sum(counts.get(score, 0) for score in [8, 9, 10])
print(f"{100 * high_count / len(train_labels):.1f}%")

Training-label counts:
Score 1: 0
Score 2: 0
Score 3: 0
Score 4: 3
Score 5: 4
Score 6: 8
Score 7: 22
Score 8: 54
Score 9: 65
Score 10: 86

Percentage with scores 8–10:
84.7%


In [52]:

import inspect
from transformers import Trainer

print("Trainer class:", type(trainer).__name__)
print("Custom compute_loss:",
      type(trainer).compute_loss is not Trainer.compute_loss)

print("\nModel loss configuration:")
print("Problem type:", trainer.model.config.problem_type)
print("Number of labels:", trainer.model.config.num_labels)

print("\nTraining arguments:")
print("Epochs:", trainer.args.num_train_epochs)
print("Learning rate:", trainer.args.learning_rate)
print("Weight decay:", trainer.args.weight_decay)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

Trainer class: Trainer
Custom compute_loss: False

Model loss configuration:
Problem type: regression
Number of labels: 1

Training arguments:
Epochs: 5
Learning rate: 2e-05
Weight decay: 0.01
Best checkpoint: /content/judgebert_lmp_results/checkpoint-155


In [53]:

import numpy as np
from collections import Counter

train_labels = np.array([
    float(item["labels"]) for item in trainer.train_dataset
])

counts = Counter(train_labels.astype(int))

# Inverse-frequency weights, tempered with square root and capped.
# Rare scores receive more attention, but not extreme influence.
class_weights = {
    score: min(
        2.0,
        np.sqrt(len(train_labels) / (10 * counts.get(score, 0)))
    )
    if counts.get(score, 0) > 0 else 1.0
    for score in range(1, 11)
}

# Normalize weights to have a training-example-weighted mean of 1.
raw_weights = np.array([
    class_weights[int(score)] for score in train_labels
])
normalizer = raw_weights.mean()
class_weights = {
    score: weight / normalizer
    for score, weight in class_weights.items()
}

print("Proposed training-only weights:")
for score in range(1, 11):
    print(f"Score {score}: {class_weights[score]:.3f}")

Proposed training-only weights:
Score 1: 1.404
Score 2: 1.404
Score 3: 1.404
Score 4: 2.807
Score 5: 2.807
Score 6: 2.441
Score 7: 1.472
Score 8: 0.940
Score 9: 0.856
Score 10: 0.745


In [54]:

import numpy as np
from collections import Counter

train_labels = np.array([
    float(item["labels"]) for item in trainer.train_dataset
])

counts = Counter(train_labels.astype(int))
n = len(train_labels)

# Conservative inverse-frequency weighting.
raw_weights = {
    score: np.sqrt(n / (10 * counts[score]))
    if counts[score] > 0 else 1.0
    for score in range(1, 11)
}

# Cap extreme weights, then normalize over training examples.
raw_weights = {
    score: np.clip(weight, 0.7, 1.5)
    for score, weight in raw_weights.items()
}

example_weights = np.array([
    raw_weights[int(score)] for score in train_labels
])
normalizer = example_weights.mean()

class_weights = {
    score: float(weight / normalizer)
    for score, weight in raw_weights.items()
}

print("Conservative weights:")
for score in range(1, 11):
    print(f"Score {score}: {class_weights[score]:.3f}")

print("Training-example-weighted average:",
      round(np.mean([class_weights[int(y)] for y in train_labels]), 4))

Conservative weights:
Score 1: 1.280
Score 2: 1.280
Score 3: 1.280
Score 4: 1.920
Score 5: 1.920
Score 6: 1.920
Score 7: 1.342
Score 8: 0.896
Score 9: 0.896
Score 10: 0.896
Training-example-weighted average: 1.0


In [55]:

import numpy as np

train_labels = np.array([
    float(item["labels"]) for item in trainer.train_dataset
])

raw_weights = {
    score: (
        1.5 if score in [4, 5, 6]
        else 1.0 if score == 7
        else 0.85 if score in [8, 9, 10]
        else 1.0
    )
    for score in range(1, 11)
}

example_weights = np.array([
    raw_weights[int(score)] for score in train_labels
])
normalizer = example_weights.mean()

class_weights = {
    score: raw_weights[score] / normalizer
    for score in range(1, 11)
}

print("Final experimental weights:")
for score in range(1, 11):
    print(f"Score {score}: {class_weights[score]:.3f}")

print("Weighted average across training examples:",
      round(np.mean([class_weights[int(y)] for y in train_labels]), 4))

Final experimental weights:
Score 1: 1.106
Score 2: 1.106
Score 3: 1.106
Score 4: 1.659
Score 5: 1.659
Score 6: 1.659
Score 7: 1.106
Score 8: 0.940
Score 9: 0.940
Score 10: 0.940
Weighted average across training examples: 1.0


In [57]:

from transformers import DataCollatorWithPadding, AutoTokenizer

# Tokenizer must match xlm-roberta-base.
weighted_tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

weighted_collator = DataCollatorWithPadding(
    tokenizer=weighted_tokenizer,
    padding=True,
)

# Rebuild the experimental trainer with dynamic padding.
weighted_trainer = WeightedRegressionTrainer(
    model=weighted_model,
    args=weighted_args,
    train_dataset=trainer.train_dataset,
    eval_dataset=trainer.eval_dataset,
    data_collator=weighted_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

weighted_trainer.train()

Epoch,Training Loss,Validation Loss,Rmse,Mae,Pearson,Spearman
1,No log,24.851049,5.105205,4.913926,0.011366,0.141248
2,No log,8.308526,2.950412,2.711392,-0.380510,-0.330927
3,No log,2.567112,1.530844,1.342024,0.476989,0.242095
4,No log,1.972669,1.268604,1.062137,0.589986,0.283997
5,No log,1.898769,1.231034,0.996639,0.590952,0.319592


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=155, training_loss=17.32769972278226, metrics={'train_runtime': 493.8043, 'train_samples_per_second': 2.45, 'train_steps_per_second': 0.314, 'total_flos': 64891236945816.0, 'train_loss': 17.32769972278226, 'epoch': 5.0})

In [58]:

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

# Use the same validation dataset for both models.
val_output = weighted_trainer.predict(trainer.eval_dataset)
weighted_true = np.asarray(val_output.label_ids).reshape(-1)
weighted_pred = np.asarray(val_output.predictions).reshape(-1)

print("Weighted validation prediction distribution:")
print("Mean:", round(float(weighted_pred.mean()), 4))
print("Std:", round(float(weighted_pred.std()), 4))
print("Min:", round(float(weighted_pred.min()), 4))
print("Max:", round(float(weighted_pred.max()), 4))

print("\nWeighted predictions rounded to integers:")
print(dict(zip(
    *np.unique(np.rint(weighted_pred).astype(int), return_counts=True)
)))

print("\nLargest weighted-model errors:")
errors = np.abs(weighted_true - weighted_pred)
for idx in np.argsort(errors)[-10:][::-1]:
    print(
        f"True={weighted_true[idx]:.0f}, "
        f"Pred={weighted_pred[idx]:.2f}, "
        f"Error={errors[idx]:.2f}"
    )

Weighted validation prediction distribution:
Mean: 8.6365
Std: 0.2682
Min: 7.3047
Max: 8.8281

Weighted predictions rounded to integers:
{np.int64(7): np.int64(1), np.int64(8): np.int64(4), np.int64(9): np.int64(38)}

Largest weighted-model errors:
True=4, Pred=7.30, Error=3.30
True=6, Pred=8.77, Error=2.77
True=6, Pred=8.69, Error=2.69
True=7, Pred=8.77, Error=1.77
True=7, Pred=8.72, Error=1.72
True=7, Pred=8.58, Error=1.58
True=10, Pred=8.51, Error=1.49
True=10, Pred=8.55, Error=1.45
True=10, Pred=8.59, Error=1.41
True=10, Pred=8.62, Error=1.38


In [59]:

import os
import pandas as pd

path = "/content/lmp_qualitative_case_analysis.csv"

print("File exists:", os.path.exists(path))

if os.path.exists(path):
    cases = pd.read_csv(path)
    print("Shape:", cases.shape)
    print("Columns:", cases.columns.tolist())
    display(cases.head(3))
else:
    print("The qualitative-analysis CSV has not been saved at this path.")

File exists: True
Shape: (15, 11)
Columns: ['ID', 'Language', 'Domain', 'Original', 'Simplified', 'LMP', 'Predicted_LMP', 'Absolute_Error', 'Legal_Error_Type', 'Legal_Impact', 'Reviewer_Explanation']


,ID,Language,Domain,Original,Simplified,LMP,Predicted_LMP,Absolute_Error,Legal_Error_Type,Legal_Impact,Reviewer_Explanation
0,hi_tos_001,hi,tos_privacy,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेवाओं का उपयोग करते समय सहमत होते हैं.",उपयोग करते समय सहमत होने वाले नियमों का वर्णन करता है.,5.0,9.10,4.10,Scope/detail omission,Low to moderate,The explicit reference to using the services is omitted.
1,hi_tos_062,hi,tos_privacy,"हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव करके रख सकते हैं, अगर हमें लगता है कि यह, धोखाधड़ी या भविष्य में गलत इस्तेमाल को रोकने के लिए ज़रूरी है या किसी कानूनी/नियामक कार्यवाही को अधूरी रहने या उस बारे में कोई कानूनी और/या नियामक निर्देश मिलने की स्थिति में कानून के हिसाब से ज़रूरी हो।","हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धोखाधड़ी रोकने या कानूनी कार्यवाही को अधूरा नहीं रहने के लिए ज़रूरी हो।",5.0,8.99,3.99,Condition/purpose omission,High,Future misuse prevention and some legal or regulatory circumstances are omitted.
2,en_rental_046,en,rental,The Landlord may rel et the Premises or any part of the Premises for a term or terms which may be less or greater than the balance of the Term remaining and may grant reasonable concessions in connection with such reletting including any alterations and improvements to the Premises.,"The Landlord may sublet the Premises or part of the Premises for a term less or greater than the remaining Term and may offer reasonable concessions, including alterations and improvements to the Premises.",4.0,7.93,3.93,Potential terminology change,Needs legal review,Relet and sublet may not be legally equivalent in this lease context.


In [60]:

import pandas as pd
import os

path = "/content/lmp_qualitative_case_analysis.csv"

if os.path.exists(path):
    cases = pd.read_csv(path)

    print("Shape:", cases.shape)
    print("\nColumns:")
    print(cases.columns.tolist())

    review_cols = [
        "Legal_Error_Type",
        "Legal_Impact",
        "Reviewer_Explanation"
    ]

    for col in review_cols:
        if col in cases.columns:
            print(f"\nMissing {col}:", cases[col].isna().sum())

    display(cases)
else:
    print("CSV not found:", path)

Shape: (15, 11)

Columns:
['ID', 'Language', 'Domain', 'Original', 'Simplified', 'LMP', 'Predicted_LMP', 'Absolute_Error', 'Legal_Error_Type', 'Legal_Impact', 'Reviewer_Explanation']

Missing Legal_Error_Type: 0

Missing Legal_Impact: 0

Missing Reviewer_Explanation: 0


,ID,Language,Domain,Original,Simplified,LMP,Predicted_LMP,Absolute_Error,Legal_Error_Type,Legal_Impact,Reviewer_Explanation
0,hi_tos_001,hi,tos_privacy,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेवाओं का उपयोग करते समय सहमत होते हैं.",उपयोग करते समय सहमत होने वाले नियमों का वर्णन करता है.,5.0,9.10,4.10,Scope/detail omission,Low to moderate,The explicit reference to using the services is omitted.
1,hi_tos_062,hi,tos_privacy,"हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव करके रख सकते हैं, अगर हमें लगता है कि यह, धोखाधड़ी या भविष्य में गलत इस्तेमाल को रोकने के लिए ज़रूरी है या किसी कानूनी/नियामक कार्यवाही को अधूरी रहने या उस बारे में कोई कानूनी और/या नियामक निर्देश मिलने की स्थिति में कानून के हिसाब से ज़रूरी हो।","हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धोखाधड़ी रोकने या कानूनी कार्यवाही को अधूरा नहीं रहने के लिए ज़रूरी हो।",5.0,8.99,3.99,Condition/purpose omission,High,Future misuse prevention and some legal or regulatory circumstances are omitted.
2,en_rental_046,en,rental,The Landlord may rel et the Premises or any part of the Premises for a term or terms which may be less or greater than the balance of the Term remaining and may grant reasonable concessions in connection with such reletting including any alterations and improvements to the Premises.,"The Landlord may sublet the Premises or part of the Premises for a term less or greater than the remaining Term and may offer reasonable concessions, including alterations and improvements to the Premises.",4.0,7.93,3.93,Potential terminology change,Needs legal review,Relet and sublet may not be legally equivalent in this lease context.
3,en_rental_057,en,rental,"Within 30 days of the termination of this lease (and Tenant vacating the Property), Landlord will provide the Tenants with an itemized statement showing any retention of any portions of the Security Deposit amount for the cost of repair or cleaning above and beyond the ordinary wear and tear on the Property or any other sums due to Landlord under this Residential Lease Agreement.","Within 30 days after the lease ends and Tenant moves out, Landlord will give Tenants a list showing any part of the Security Deposit kept for repairs or cleaning beyond normal wear and tear or other amounts owed under this agreement.",4.0,7.61,3.61,Condition/detail omission,Moderate,The explicit tenant-vacating condition and some itemization detail are omitted.
4,en_rental_035,en,rental,It is the intent of this Lease and agreed to by the Parties to this Lease that rent for this Lease will be on a gross rent basis meaning the Tenant will pay the Base Rent and any Additional Rent and the Landlord will be responsible for all other service charges related to the Premises and the operation of the Building save as specifically provided in this Lease to the contrary.,"It is agreed that the Tenant will pay the Base Rent and any Additional Rent, and the Landlord will cover all other service charges related to the Premises and the Building, except as specifically stated otherwise in this Lease.",4.0,7.60,3.60,No material change apparent,Low,The gross-rent arrangement and exception are substantially retained.
5,en_rental_038,en,rental,"Should the Tenant remain in possession of the Premises with the consent of the Landlord after the natural expiration of this Lease, a new tenancy from month to month will be created between the Landlord and the Tenant which will be subject to all the terms and conditions of this Lease but will be terminable upon either party giving one month's notice to the other party.","If the Tenant stays in the Premises with the Landlord's permission after the lease ends, a month-to-month tenancy will be created. This tenancy will follow all the terms of the original lease and can be ended by either party giving one month's notice.",4.0,7.34,3.34,No material change apparent,Low,"The tenancy type, original terms, and one-month notice are retained."
6,hi_tos_076,hi,tos_privacy,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाणीकरण सुनि

In [61]:

import pandas as pd

path = "/content/lmp_qualitative_case_analysis.csv"
cases = pd.read_csv(path)

cols = ["ID", "LMP", "Predicted_LMP", "Absolute_Error"]

print("Missing values:")
print(cases[cols].isna().sum())

print("\nScore values:")
print(cases[cols].to_string(index=False))

print("\nData types:")
print(cases[cols].dtypes)

Missing values:
ID                0
LMP               0
Predicted_LMP     0
Absolute_Error    0
dtype: int64

Score values:
           ID  LMP  Predicted_LMP  Absolute_Error
   hi_tos_001  5.0           9.10            4.10
   hi_tos_062  5.0           8.99            3.99
en_rental_046  4.0           7.93            3.93
en_rental_057  4.0           7.61            3.61
en_rental_035  4.0           7.60            3.60
en_rental_038  4.0           7.34            3.34
   hi_tos_076  6.0           9.13            3.13
   hi_tos_061  6.0           9.12            3.12
   en_tos_117  5.0           8.11            3.11
hi_rental_047  6.0           9.09            3.09
   hi_tos_018  6.0           9.09            3.09
   hi_tos_029  6.0           9.08            3.08
   hi_tos_034  6.0           9.05            3.05
   en_tos_144  6.0           8.98            2.98
   en_tos_126  6.0           8.84            2.84

Data types:
ID                 object
LMP               float64
Predicted_L

In [62]:

import os
import pandas as pd

files = {
    "Model comparison": "/content/lmp_model_comparison_clean.csv",
    "Held-out domain": "/content/held_out_domain_results.csv",
    "Sanity checks": "/content/lmp_sanity_check_results.csv",
    "Qualitative analysis": "/content/lmp_qualitative_case_analysis.csv",
}

for name, path in files.items():
    print(f"\n{'=' * 15} {name} {'=' * 15}")

    if os.path.exists(path):
        result = pd.read_csv(path)
        print("Shape:", result.shape)
        print("Columns:", result.columns.tolist())
        display(result)
    else:
        print("FILE NOT FOUND:", path)


=============== Model comparison ===============
Shape: (3, 5)
Columns: ['Model', 'Pearson', 'Spearman', 'RMSE', 'MAE']


,Model,Pearson,Spearman,RMSE,MAE
0,TF-IDF,0.5976,0.7100,1.1634,0.7924
1,MiniLM,0.5875,0.6574,1.1058,0.8477
2,XLM-R,0.5784,0.4195,1.2325,0.9754



=============== Held-out domain ===============
Shape: (2, 9)
Columns: ['Train_Domain', 'Test_Domain', 'Train_N', 'Validation_N', 'Test_N', 'RMSE', 'MAE', 'Pearson', 'Spearman']


,Train_Domain,Test_Domain,Train_N,Validation_N,Test_N,RMSE,MAE,Pearson,Spearman
0,rental,tos_privacy,110,20,227,3.666976,3.459630,0.028062,-0.001710
1,tos_privacy,rental,192,35,130,1.761515,1.539754,0.309955,0.301214



=============== Sanity checks ===============
Shape: (4, 7)
Columns: ['Case', 'Original', 'Simplified', 'TFIDF_LMP', 'MiniLM_Cosine', 'MiniLM_LMP', 'XLMR_LMP']


,Case,Original,Simplified,TFIDF_LMP,MiniLM_Cosine,MiniLM_LMP,XLMR_LMP
0,Identical,The tenant must pay rent on time.,The tenant must pay rent on time.,9.32,1.0000,9.32,8.95
1,Identical,We store personal information in India.,We store personal information in India.,9.32,1.0000,9.32,8.91
2,Unrelated,The tenant must pay rent on time.,The Earth revolves around the Sun.,1.76,0.0338,1.00,8.55
3,Unrelated,We store personal information in India.,The recipe requires two cups of flour.,1.00,-0.1198,1.00,8.72



=============== Qualitative analysis ===============
Shape: (15, 11)
Columns: ['ID', 'Language', 'Domain', 'Original', 'Simplified', 'LMP', 'Predicted_LMP', 'Absolute_Error', 'Legal_Error_Type', 'Legal_Impact', 'Reviewer_Explanation']


,ID,Language,Domain,Original,Simplified,LMP,Predicted_LMP,Absolute_Error,Legal_Error_Type,Legal_Impact,Reviewer_Explanation
0,hi_tos_001,hi,tos_privacy,"उन नियमों का वर्णन करता है, जिनसे आप हमारी सेवाओं का उपयोग करते समय सहमत होते हैं.",उपयोग करते समय सहमत होने वाले नियमों का वर्णन करता है.,5.0,9.10,4.10,Scope/detail omission,Low to moderate,The explicit reference to using the services is omitted.
1,hi_tos_062,hi,tos_privacy,"हमसे जुड़ी निजी जानकारी को उस स्थिति में सेव करके रख सकते हैं, अगर हमें लगता है कि यह, धोखाधड़ी या भविष्य में गलत इस्तेमाल को रोकने के लिए ज़रूरी है या किसी कानूनी/नियामक कार्यवाही को अधूरी रहने या उस बारे में कोई कानूनी और/या नियामक निर्देश मिलने की स्थिति में कानून के हिसाब से ज़रूरी हो।","हमसे जुड़ी निजी जानकारी को रख सकते हैं, यदि धोखाधड़ी रोकने या कानूनी कार्यवाही को अधूरा नहीं रहने के लिए ज़रूरी हो।",5.0,8.99,3.99,Condition/purpose omission,High,Future misuse prevention and some legal or regulatory circumstances are omitted.
2,en_rental_046,en,rental,The Landlord may rel et the Premises or any part of the Premises for a term or terms which may be less or greater than the balance of the Term remaining and may grant reasonable concessions in connection with such reletting including any alterations and improvements to the Premises.,"The Landlord may sublet the Premises or part of the Premises for a term less or greater than the remaining Term and may offer reasonable concessions, including alterations and improvements to the Premises.",4.0,7.93,3.93,Potential terminology change,Needs legal review,Relet and sublet may not be legally equivalent in this lease context.
3,en_rental_057,en,rental,"Within 30 days of the termination of this lease (and Tenant vacating the Property), Landlord will provide the Tenants with an itemized statement showing any retention of any portions of the Security Deposit amount for the cost of repair or cleaning above and beyond the ordinary wear and tear on the Property or any other sums due to Landlord under this Residential Lease Agreement.","Within 30 days after the lease ends and Tenant moves out, Landlord will give Tenants a list showing any part of the Security Deposit kept for repairs or cleaning beyond normal wear and tear or other amounts owed under this agreement.",4.0,7.61,3.61,Condition/detail omission,Moderate,The explicit tenant-vacating condition and some itemization detail are omitted.
4,en_rental_035,en,rental,It is the intent of this Lease and agreed to by the Parties to this Lease that rent for this Lease will be on a gross rent basis meaning the Tenant will pay the Base Rent and any Additional Rent and the Landlord will be responsible for all other service charges related to the Premises and the operation of the Building save as specifically provided in this Lease to the contrary.,"It is agreed that the Tenant will pay the Base Rent and any Additional Rent, and the Landlord will cover all other service charges related to the Premises and the Building, except as specifically stated otherwise in this Lease.",4.0,7.60,3.60,No material change apparent,Low,The gross-rent arrangement and exception are substantially retained.
5,en_rental_038,en,rental,"Should the Tenant remain in possession of the Premises with the consent of the Landlord after the natural expiration of this Lease, a new tenancy from month to month will be created between the Landlord and the Tenant which will be subject to all the terms and conditions of this Lease but will be terminable upon either party giving one month's notice to the other party.","If the Tenant stays in the Premises with the Landlord's permission after the lease ends, a month-to-month tenancy will be created. This tenancy will follow all the terms of the original lease and can be ended by either party giving one month's notice.",4.0,7.34,3.34,No material change apparent,Low,"The tenancy type, original terms, and one-month notice are retained."
6,hi_tos_076,hi,tos_privacy,PhonePe को आपकी पहचान की पुष्टि करने और प्रमाणीकरण सुनि

In [63]:

import pandas as pd

cases = pd.read_csv("/content/lmp_qualitative_case_analysis.csv")
comparison = pd.read_csv("/content/lmp_model_comparison_clean.csv")

print("QUALITATIVE REVIEW")
print("Total cases:", len(cases))
print("Missing review fields:")
print(cases[
    ["Legal_Error_Type", "Legal_Impact", "Reviewer_Explanation"]
].isna().sum())

print("\nCases with blank original or simplified text:")
print(cases[
    cases["Original"].isna() | cases["Simplified"].isna()
][["ID", "Original", "Simplified"]].to_string(index=False))

print("\nMODEL COMPARISON")
print(comparison.to_string(index=False))

print("\nSaved files:")
for path in [
    "/content/lmp_model_comparison_clean.csv",
    "/content/held_out_domain_results.csv",
    "/content/lmp_sanity_check_results.csv",
    "/content/lmp_qualitative_case_analysis.csv",
    "/content/lmp_prediction_results.csv",
]:
    import os
    print(f"{os.path.basename(path)}: {os.path.exists(path)}")

QUALITATIVE REVIEW
Total cases: 15
Missing review fields:
Legal_Error_Type        0
Legal_Impact            0
Reviewer_Explanation    0
dtype: int64

Cases with blank original or simplified text:
Empty DataFrame
Columns: [ID, Original, Simplified]
Index: []

MODEL COMPARISON
 Model  Pearson  Spearman   RMSE    MAE
TF-IDF   0.5976    0.7100 1.1634 0.7924
MiniLM   0.5875    0.6574 1.1058 0.8477
 XLM-R   0.5784    0.4195 1.2325 0.9754

Saved files:
lmp_model_comparison_clean.csv: True
held_out_domain_results.csv: True
lmp_sanity_check_results.csv: True
lmp_qualitative_case_analysis.csv: True
lmp_prediction_results.csv: True


In [64]:

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

path = "/content/tfidf_clean_test_predictions.csv"
preds = pd.read_csv(path)

print("Shape:", preds.shape)
print("Columns:", preds.columns.tolist())
display(preds.head())

print("\nNumeric columns:")
print(preds.select_dtypes(include="number").columns.tolist())

Shape: (72, 8)
Columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP', 'TFIDF_Cosine', 'Predicted_LMP']


,ID,Original,Simplified,Language,Domain,LMP,TFIDF_Cosine,Predicted_LMP
0,hi_tos_018,दूसरों को या खुद को नुकसान न पहुंचाना या दूसरों के साथ या अपने साथ बुरा बर्ताव न करना (या न तो ऐसे नुकसान या बुरे बर्ताव की धमकी देना या इन्हें बढ़ावा देना).,दूसरों को या खुद को नुकसान न पहुंचाना और बुरा बर्ताव न करना।,hi,tos_privacy,6.0,0.782214,9.167916
1,en_tos_043,"If we are unable to resolve a dispute within sixty (60) days after the Notice of Dispute is received, you or we may commence arbitration.","If we cannot resolve a dispute within 60 days after receiving the Notice of Dispute, you or we can start arbitration.",en,tos_privacy,9.0,0.807597,9.276008
2,hi_rental_006,अनुबंध समाप्त करने के लिए \_\_\_\_\_\_ पूर्व लिखित सूचना देनी होगी।,अनुबंध समाप्त करने के लिए पूर्व लिखित सूचना देनी होगी।,hi,rental,10.0,1.000000,10.000000
3,hi_rental_015,बातचीत से समाधान न होने पर \_\_\_\_\_\_ के सक्षम न्यायालय में कार्यवाही की जा सकती है।,बातचीत से समाधान न होने पर कार्यवाही की जा सकती है \_\_\_\_\_\_ के सक्षम न्यायालय में।,hi,rental,8.0,0.937295,9.828324
4,en_tos_057,These business or user terms and conditions are available on the respective Participating Platforms.,These terms and conditions are available on the respective Participating Platforms.,en,tos_privacy,9.0,0.862690,9.510620



Numeric columns:
['LMP', 'TFIDF_Cosine', 'Predicted_LMP']


In [65]:

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error

preds = pd.read_csv("/content/tfidf_clean_test_predictions.csv")

y_true = pd.to_numeric(preds["LMP"], errors="coerce")
y_pred = pd.to_numeric(preds["Predicted_LMP"], errors="coerce")

valid = y_true.notna() & y_pred.notna()
y_true = y_true[valid].to_numpy()
y_pred = y_pred[valid].to_numpy()

print("Valid rows:", len(y_true))
print("Missing/invalid rows:", len(preds) - len(y_true))
print("Pearson:", round(float(pearsonr(y_true, y_pred).statistic), 4))
print("Spearman:", round(float(spearmanr(y_true, y_pred).statistic), 4))
print("RMSE:", round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4))
print("MAE:", round(float(mean_absolute_error(y_true, y_pred)), 4))

Valid rows: 72
Missing/invalid rows: 0
Pearson: 0.5976
Spearman: 0.71
RMSE: 1.1634
MAE: 0.7924


In [66]:

import numpy as np

print("=== SPLIT INDEX CHECK ===")

required = ["train_idx", "val_idx", "test_idx"]

if all(name in globals() for name in required):
    train_set = set(map(int, train_idx))
    val_set = set(map(int, val_idx))
    test_set = set(map(int, test_idx))

    print("Train:", len(train_set))
    print("Validation:", len(val_set))
    print("Test:", len(test_set))
    print("Train/validation overlap:", len(train_set & val_set))
    print("Train/test overlap:", len(train_set & test_set))
    print("Validation/test overlap:", len(val_set & test_set))
    print("Total unique indexes:", len(train_set | val_set | test_set))
else:
    print("Split index variables missing:", [
        name for name in required if name not in globals()
    ])

print("\n=== TRAINER DATASETS ===")

if "trainer" in globals():
    print("Trainer class:", type(trainer).__name__)
    print("Train dataset rows:", len(trainer.train_dataset))
    print("Validation dataset rows:", len(trainer.eval_dataset))
    print("Training arguments:")
    for key in [
        "num_train_epochs",
        "learning_rate",
        "seed",
        "data_seed",
        "load_best_model_at_end",
        "metric_for_best_model",
    ]:
        print(f"{key}:", getattr(trainer.args, key, "Not available"))
else:
    print("Variable 'trainer' not found.")

print("\n=== SAVED XLM-R OUTPUT ===")

import os
path = "/content/xlmr_test_predictions_audit.csv"

if os.path.exists(path):
    import pandas as pd
    saved = pd.read_csv(path)
    print("Saved test predictions:", len(saved))
    print("Columns:", saved.columns.tolist())
    print("Unique test IDs:", saved["ID"].nunique() if "ID" in saved else "ID column missing")
else:
    print("Audit prediction file not found:", path)

=== SPLIT INDEX CHECK ===
Train: 285
Validation: 43
Test: 72
Train/validation overlap: 43
Train/test overlap: 0
Validation/test overlap: 0
Total unique indexes: 357

=== TRAINER DATASETS ===
Trainer class: Trainer
Train dataset rows: 242
Validation dataset rows: 43
Training arguments:
num_train_epochs: 5
learning_rate: 2e-05
seed: 42
data_seed: None
load_best_model_at_end: True
metric_for_best_model: eval_rmse

=== SAVED XLM-R OUTPUT ===
Saved test predictions: 72
Columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP', 'Predicted_LMP', 'Preservation_Decision']
Unique test IDs: 72


In [67]:

import pandas as pd
import numpy as np

# Reconstruct the intended split from the full model dataset.
all_idx = np.arange(len(model_df))

from sklearn.model_selection import train_test_split

expected_train_idx, expected_test_idx = train_test_split(
    all_idx, test_size=0.20, random_state=42
)
expected_train_idx, expected_val_idx = train_test_split(
    expected_train_idx, test_size=0.15, random_state=42
)

expected_train_ids = set(model_df.iloc[expected_train_idx]["ID"].astype(str))
expected_val_ids = set(model_df.iloc[expected_val_idx]["ID"].astype(str))
expected_test_ids = set(model_df.iloc[expected_test_idx]["ID"].astype(str))

saved_test = pd.read_csv("/content/xlmr_test_predictions_audit.csv")
saved_test_ids = set(saved_test["ID"].astype(str))

print("=== RECONSTRUCTED SPLIT ===")
print("Train:", len(expected_train_ids))
print("Validation:", len(expected_val_ids))
print("Test:", len(expected_test_ids))
print("Expected test IDs match saved predictions:",
      expected_test_ids == saved_test_ids)

print("\n=== TRAINER DATASET STRUCTURE ===")
print("Train columns:", trainer.train_dataset.column_names)
print("Validation columns:", trainer.eval_dataset.column_names)

print("\n=== ORIGINAL TRAINING DATA EVIDENCE ===")
if "train_df" in globals():
    print("train_df rows:", len(train_df))
    print("train_df columns:", train_df.columns.tolist())
    if "ID" in train_df.columns:
        actual_train_ids = set(train_df["ID"].astype(str))
        print("train_df IDs match expected training IDs:",
              actual_train_ids == expected_train_ids)
else:
    print("No train_df variable found; actual training IDs are not directly available.")

=== RECONSTRUCTED SPLIT ===
Train: 242
Validation: 43
Test: 72
Expected test IDs match saved predictions: True

=== TRAINER DATASET STRUCTURE ===
Train columns: ['Original', 'Simplified', 'labels', 'input_ids', 'attention_mask']
Validation columns: ['Original', 'Simplified', 'labels', 'input_ids', 'attention_mask']

=== ORIGINAL TRAINING DATA EVIDENCE ===
train_df rows: 242
train_df columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP']
train_df IDs match expected training IDs: True


In [68]:
print("train_df matches expected split:",
      set(train_df["ID"].astype(str)) == expected_train_ids)

print("val_df available:", "val_df" in globals())
if "val_df" in globals() and "ID" in val_df.columns:
    print("val_df rows:", len(val_df))
    print("val_df matches expected split:",
          set(val_df["ID"].astype(str)) == expected_val_ids)

print("\nTrainer train rows:", len(trainer.train_dataset))
print("Trainer validation rows:", len(trainer.eval_dataset))

print("\nTraining history:")
for item in trainer.state.log_history:
    if "epoch" in item and (
        "eval_rmse" in item or "loss" in item
    ):
        print(item)

train_df matches expected split: True
val_df available: True
val_df rows: 43
val_df matches expected split: True

Trainer train rows: 242
Trainer validation rows: 43

Training history:
{'loss': 61.921244959677416, 'grad_norm': 727.2415771484375, 'learning_rate': 1.6129032258064517e-05, 'epoch': 1.0, 'step': 31}
{'eval_loss': 26.063982009887695, 'eval_rmse': 5.105289610775054, 'eval_mae': 4.921238899230957, 'eval_pearson': 0.11395490169525146, 'eval_spearman': 0.13696668188141883, 'eval_runtime': 0.2099, 'eval_samples_per_second': 204.865, 'eval_steps_per_second': 28.586, 'epoch': 1.0, 'step': 31}
{'loss': 20.119991179435484, 'grad_norm': 138.05906677246094, 'learning_rate': 1.2129032258064518e-05, 'epoch': 2.0, 'step': 62}
{'eval_loss': 6.526846885681152, 'eval_rmse': 2.5547691588810286, 'eval_mae': 2.3210391998291016, 'eval_pearson': -0.3471970558166504, 'eval_spearman': -0.36035296616527684, 'eval_runtime': 0.3317, 'eval_samples_per_second': 129.625, 'eval_steps_per_second': 18.087, 

In [69]:
cd /path/to/your/project
git status --short
git branch --show-current
ls -la
find . -maxdepth 2 -type f \
  -not -path './.git/*' \
  -not -path './.venv/*' \
  -not -path './venv/*' | sort | head -200

SyntaxError: invalid syntax (3334773135.py, line 2)

In [70]:
import os

print("Current directory:", os.getcwd())
print("\nFiles in current directory:")
print(os.listdir("."))

Current directory: /content

Files in current directory:
['.config', 'domain_exp_tos_privacy_to_rental', 'lmp_final - lmp_semantic_review(1).csv', 'judgebert_lmp_results', 'lmp_qualitative_case_analysis.csv', 'xlmr_test_predictions_audit.csv', 'lmp_split_audit.csv', 'judgebert_lmp_weighted', 'generalization_by_domain.csv', 'lmp_sanity_check_results.csv', 'lmp_sanity_checks.csv', 'judgebert_lmp_final', 'minilm_clean_test_predictions.csv', 'domain_exp_rental_to_tos_privacy', 'held_out_domain_results.csv', 'lmp_model_comparison_clean.csv', 'tfidf_clean_test_predictions.csv', 'lmp_prediction_results.csv', 'generalization_by_language.csv', 'sample_data']


In [71]:
!git status --short
!git branch --show-current
!ls -la

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
total 528
drwxr-xr-x 1 root root   4096 Oct  9 18:00  .
drwxr-xr-x 1 root root   4096 Oct  9 17:00  ..
drwxr-xr-x 4 root root   4096 Sep 30 13:28  .config
drwxr-xr-x 3 root root   4096 Oct  9 17:33  domain_exp_rental_to_tos_privacy
drwxr-xr-x 3 root root   4096 Oct  9 17:40  domain_exp_tos_privacy_to_rental
-rw-r--r-- 1 root root    187 Oct  9 17:24  generalization_by_domain.csv
-rw-r--r-- 1 root root    177 Oct  9 17:24  generalization_by_language.csv
-rw-r--r-- 1 root root    294 Oct  9 17:40  held_out_domain_results.csv
drwxr-xr-x 2 root root   4096 Oct  9 17:17  judgebert_lmp_final
drwxr-xr-x 7 root root   4096 Oct  9 17:15  judgebert_lmp_results
drwxr-xr-x 7 root root   4096 Oct  9 18:07  judgebert_lmp_weighted
-rw-r--r-- 1 root root 214669 Oct  9 17:08 'lmp_final - lmp_semantic_review(1).csv'
-rw-r--r-- 1 root root    134 Oct  9 17:51  lmp_mode

In [73]:
import os

for folder in [
    "/content/judgebert_lmp_final",
    "/content/judgebert_lmp_results",
    "/content/judgebert_lmp_weighted",
    "/content/domain_exp_tos_privacy_to_rental",
    "/content/domain_exp_rental_to_tos_privacy",
]:
    print(f"\n--- {folder} ---")
    if os.path.isdir(folder):
        for root, dirs, files in os.walk(folder):
            dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]
            level = root.replace(folder, "").count(os.sep)
            if level > 2:
                dirs[:] = []
                continue
            print(root, "->", files[:15])
    else:
        print("Not found")


--- /content/judgebert_lmp_final ---
/content/judgebert_lmp_final -> ['model.safetensors', 'tokenizer.json', 'config.json', 'tokenizer_config.json', 'training_args.bin']

--- /content/judgebert_lmp_results ---
/content/judgebert_lmp_results -> []
/content/judgebert_lmp_results/checkpoint-62 -> ['scheduler.pt', 'trainer_state.json', 'model.safetensors', 'tokenizer.json', 'config.json', 'optimizer.pt', 'tokenizer_config.json', 'training_args.bin', 'scaler.pt', 'rng_state.pth']
/content/judgebert_lmp_results/checkpoint-93 -> ['scheduler.pt', 'trainer_state.json', 'model.safetensors', 'tokenizer.json', 'config.json', 'optimizer.pt', 'tokenizer_config.json', 'training_args.bin', 'scaler.pt', 'rng_state.pth']
/content/judgebert_lmp_results/checkpoint-155 -> ['scheduler.pt', 'trainer_state.json', 'model.safetensors', 'tokenizer.json', 'config.json', 'optimizer.pt', 'tokenizer_config.json', 'training_args.bin', 'scaler.pt', 'rng_state.pth']
/content/judgebert_lmp_results/checkpoint-124 -> ['s

In [74]:
import os

for folder in [
    "/content/judgebert_lmp_final",
    "/content/judgebert_lmp_results",
    "/content/judgebert_lmp_weighted",
]:
    print(f"\n{'=' * 15} {folder} {'=' * 15}")

    for root, dirs, files in os.walk(folder):
        dirs[:] = [
            d for d in dirs
            if d not in {".git", "__pycache__"}
        ]

        depth = root.replace(folder, "").count(os.sep)
        if depth > 2:
            dirs[:] = []
            continue

        for filename in files:
            path = os.path.join(root, filename)
            print(
                os.path.relpath(path, folder),
                f"({os.path.getsize(path) / 1024:.1f} KB)"
            )


=============== /content/judgebert_lmp_final ===============
model.safetensors (1086134.7 KB)
tokenizer.json (16697.4 KB)
config.json (0.9 KB)
tokenizer_config.json (0.3 KB)
training_args.bin (5.1 KB)

=============== /content/judgebert_lmp_results ===============
checkpoint-62/scheduler.pt (1.4 KB)
checkpoint-62/trainer_state.json (1.8 KB)
checkpoint-62/model.safetensors (1086134.7 KB)
checkpoint-62/tokenizer.json (16697.4 KB)
checkpoint-62/config.json (0.9 KB)
checkpoint-62/optimizer.pt (2172389.4 KB)
checkpoint-62/tokenizer_config.json (0.3 KB)
checkpoint-62/training_args.bin (5.1 KB)
checkpoint-62/scaler.pt (1.4 KB)
checkpoint-62/rng_state.pth (14.3 KB)
checkpoint-93/scheduler.pt (1.4 KB)
checkpoint-93/trainer_state.json (2.4 KB)
checkpoint-93/model.safetensors (1086134.7 KB)
checkpoint-93/tokenizer.json (16697.4 KB)
checkpoint-93/config.json (0.9 KB)
checkpoint-93/optimizer.pt (2172389.4 KB)
checkpoint-93/tokenizer_config.json (0.3 KB)
checkpoint-93/training_args.bin (5.1 KB)
che

In [75]:
import os

project_dir = "/content/judgebert_lmp_submission"

print("Submission folder exists:", os.path.isdir(project_dir))

if os.path.isdir(project_dir):
    for root, dirs, files in os.walk(project_dir):
        level = root.replace(project_dir, "").count(os.sep)
        if level > 2:
            dirs[:] = []
            continue
        print(root, "->", files)
else:
    print("The folder has not been created yet.")

Submission folder exists: False
The folder has not been created yet.


In [77]:
import os
import shutil

project_dir = "/content/judgebert_lmp_submission"

# Create submission folders
for folder in ["results", "notebooks"]:
    os.makedirs(os.path.join(project_dir, folder), exist_ok=True)

# Copy verified experiment outputs
result_files = [
    "lmp_model_comparison_clean.csv",
    "held_out_domain_results.csv",
    "lmp_sanity_check_results.csv",
    "lmp_qualitative_case_analysis.csv",
    "lmp_split_audit.csv",
    "tfidf_clean_test_predictions.csv",
    "minilm_clean_test_predictions.csv",
    "xlmr_test_predictions_audit.csv",
    "lmp_prediction_results.csv",
    "generalization_by_domain.csv",
    "generalization_by_language.csv",
]

copied, missing = [], []

for filename in result_files:
    source = f"/content/{filename}"
    destination = os.path.join(project_dir, "results", filename)

    if os.path.isfile(source):
        shutil.copy2(source, destination)
        copied.append(filename)
    else:
        missing.append(filename)

print("Project directory:", project_dir)
print("\nCopied files:", len(copied))
for filename in copied:
    print("✅", filename)

print("\nMissing files:", missing if missing else "None")

Project directory: /content/judgebert_lmp_submission

Copied files: 11
✅ lmp_model_comparison_clean.csv
✅ held_out_domain_results.csv
✅ lmp_sanity_check_results.csv
✅ lmp_qualitative_case_analysis.csv
✅ lmp_split_audit.csv
✅ tfidf_clean_test_predictions.csv
✅ minilm_clean_test_predictions.csv
✅ xlmr_test_predictions_audit.csv
✅ lmp_prediction_results.csv
✅ generalization_by_domain.csv
✅ generalization_by_language.csv

Missing files: None


In [78]:

from pathlib import Path

project_dir = Path("/content/judgebert_lmp_submission")

readme = r"""# Explainable Multilingual Legal Meaning Preservation

## Project Overview

This project investigates whether simplified legal text preserves the meaning of its original version in English and Hindi.

It compares two semantic similarity baselines with a JUDGEBERT-inspired XLM-RoBERTa regression model. The model predicts a Legal Meaning Preservation (LMP) score on a 1–10 scale.

**Important:** This is a JUDGEBERT-inspired approach, not an exact reproduction of the original JUDGEBERT implementation.

## Objectives

- Estimate meaning preservation between original and simplified legal sentences.
- Compare TF-IDF, Sentence Transformers (MiniLM), and XLM-RoBERTa regression.
- Evaluate generalization across legal domains.
- Test identical and unrelated sentence pairs.
- Analyze legally important changes such as omitted conditions, exceptions, scope, and retention periods.
- Produce scores, provisional preservation decisions, and qualitative explanations.

## Dataset

- Total sentence pairs: 357
- Languages: English and Hindi
- Domains: rental agreements and terms/privacy policies
- Target: supplied LMP ratings on a 1–10 scale

The dataset contains a high proportion of ratings between 8 and 10. Results should therefore be interpreted in light of label imbalance and dataset size.

Human-rating provenance and annotation agreement should be confirmed before making claims about independently validated human judgments.

## Methods

### Baselines

1. **TF-IDF:** Text representations with a linear regression calibration fitted on training data.
2. **MiniLM:** Sentence embeddings with a linear regression calibration fitted on training data.
3. **XLM-RoBERTa:** A transformer regression model trained to predict LMP ratings.

The clean baseline comparisons use the same reconstructed train/validation/test split. TF-IDF fitting and baseline calibration are restricted to training data.

### Evaluation Metrics

- Pearson correlation
- Spearman rank correlation
- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)

### Generalization Experiment

Models were evaluated on a domain excluded from their training and validation data:
- Train on rental agreements; test on terms/privacy policies.
- Train on terms/privacy policies; test on rental agreements.

### Qualitative and Sanity Checks

The project includes identical-pair and unrelated-pair checks, plus 15 manually reviewed cases involving potentially legally meaningful changes. These qualitative assessments are preliminary and are not legal advice.

## Results

### Clean Test-Set Comparison

| Model | Pearson | Spearman | RMSE | MAE |
|---|---:|---:|---:|---:|
| TF-IDF | 0.5976 | 0.7100 | 1.1634 | 0.7924 |
| MiniLM | 0.5875 | 0.6574 | 1.1058 | 0.8477 |
| XLM-RoBERTa | 0.5784 | 0.4195 | 1.2325 | 0.9754 |

TF-IDF achieved the highest Pearson and Spearman correlations and the lowest MAE. MiniLM achieved the lowest RMSE. XLM-RoBERTa did not outperform these baselines on the reported test metrics.

These are descriptive comparisons; no statistical significance testing was performed.

### Cross-Domain Generalization

| Training domain | Test domain | RMSE | MAE | Pearson | Spearman |
|---|---|---:|---:|---:|---:|
| Rental | Terms/privacy | 3.6670 | 3.4596 | 0.0281 | -0.0017 |
| Terms/privacy | Rental | 1.7615 | 1.5398 | 0.3100 | 0.3012 |

The results indicate weak cross-domain generalization, especially when training on rental agreements and testing on terms/privacy policies.

### Sanity-Check Finding

The XLM-RoBERTa model assigned high scores to some unrelated sentence pairs. This is an important limitation: a high predicted LMP score should not be treated as proof that legal meaning is preserved.

## Preservation Decisions

The prediction output uses a provisional threshold of 8.0:
- Score >= 8: `Preserved`
- Score < 8: `Review: possible meaning loss`

This threshold has not been independently validated. Decisions should be reviewed by a qualified human, especially where obligations, exceptions, conditions, dates, amounts, or legal rights may have changed.

## Repository Structure

```text
judgebert_lmp_submission/
├── README.md
├── requirements.txt
├── notebooks/
└── results/
    ├── lmp_model_comparison_clean.csv
    ├── held_out_domain_results.csv
    ├── lmp_sanity_check_results.csv
    ├── lmp_qualitative_case_analysis.csv
    ├── lmp_split_audit.csv
    ├── tfidf_clean_test_predictions.csv
    ├── minilm_clean_test_predictions.csv
    ├── xlmr_test_predictions_audit.csv
    ├── lmp_prediction_results.csv
    ├── generalization_by_domain.csv
    └── generalization_by_language.csv
```

## Reproducibility

The `notebooks/` directory is intended to contain the experiment notebook. The current results and prediction files are included for inspection.

Before claiming full reproducibility, add the complete notebook, pin the software dependencies, document the dataset source and preprocessing, and verify that the experiments can be rerun from a clean runtime.

## Limitations

- Small dataset with an imbalanced LMP label distribution.
- Performance does not establish legal correctness.
- XLM-RoBERTa produced suspiciously high scores for some unrelated pairs.
- Cross-domain generalization was weak.
- The preservation threshold is provisional.
- Qualitative reviews are preliminary and should be independently checked.
- Results are not a substitute for legal review.

## References

- GRAAL Research, JUDGEBERT: https://github.com/GRAAL-Research/JUDGEBERT
- XLM-RoBERTa: https://huggingface.co/FacebookAI/xlm-roberta-base
- Sentence Transformers: https://www.sbert.net/

## Disclaimer

This is an academic research prototype for evaluating legal meaning preservation. It is not a legal advice system and should not be used to make legal decisions without qualified human review.
"""

(project_dir / "README.md").write_text(readme, encoding="utf-8")

print("README created:", project_dir / "README.md")
print("Size:", (project_dir / "README.md").stat().st_size, "bytes")

README created: /content/judgebert_lmp_submission/README.md
Size: 5899 bytes


In [79]:
import os

print("Possible notebooks in /content:")
for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in [".config", "sample_data"]]
    for file in files:
        if file.endswith(".ipynb"):
            print(os.path.join(root, file))

Possible notebooks in /content:


In [81]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [82]:
import os

matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    dirs[:] = [
        d for d in dirs
        if d not in [".trash", ".ipynb_checkpoints"]
    ]
    for file in files:
        if file.endswith(".ipynb"):
            matches.append(os.path.join(root, file))

print("Notebook files found:", len(matches))
for path in matches:
    print(path)

Notebook files found: 1
/content/drive/MyDrive/Colab Notebooks/IMD_LAB6.ipynb


In [83]:
import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/IMD_LAB6.ipynb"
)

with notebook_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook.get("cells", [])
code_cells = [
    "".join(cell.get("source", []))
    for cell in cells
    if cell.get("cell_type") == "code"
]

all_code = "\n".join(code_cells)
all_text = "\n".join(
    "".join(cell.get("source", []))
    for cell in cells
)

print("Notebook:", notebook_path.name)
print("Total cells:", len(cells))
print("Code cells:", len(code_cells))
print("Total code characters:", len(all_code))

keywords = [
    "JUDGEBERT", "xlm-roberta", "LMP",
    "lmp_final", "TF-IDF", "MiniLM",
    "train_test_split", "Trainer"
]

print("\nProject keyword check:")
for keyword in keywords:
    print(f"{keyword}:",
          keyword.lower() in all_text.lower())

print("\nFirst 15 cell previews:")
for i, cell in enumerate(cells[:15]):
    source = "".join(cell.get("source", []))
    print(f"\n--- Cell {i + 1} ({cell.get('cell_type')}) ---")
    print(source[:350].replace("\n", " "))

Notebook: IMD_LAB6.ipynb
Total cells: 19
Code cells: 18
Total code characters: 22800

Project keyword check:
JUDGEBERT: False
xlm-roberta: False
LMP: False
lmp_final: False
TF-IDF: False
MiniLM: False
train_test_split: False
Trainer: False

First 15 cell previews:

--- Cell 1 (markdown) ---


--- Cell 2 (code) ---
# !pip install torch nltk pandas matplotlib      # uncomment if running in a fresh environment import os, re, random, math, time, urllib.request from collections import Counter  import numpy as np import pandas as pd import torch import torch.nn as nn import torch.nn.functional as F from torch.utils.data import TensorDataset, DataLoader from torch.

--- Cell 3 (code) ---
# ---------------- Configuration (change here) ---------------- DATA_URL   = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/english-french-both.csv" DATA_FILE  = "eng-fra.csv" N_SAMPLES  = 40000    # number of sentence pairs used (keeps the lab fast); set None to use everything MAX_LEN    = 10  

In [84]:
import os
import zipfile

project_dir = "/content/judgebert_lmp_submission"
zip_path = "/content/judgebert_lmp_submission.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(project_dir):
        for filename in files:
            full_path = os.path.join(root, filename)
            relative_path = os.path.relpath(full_path, "/content")
            zf.write(full_path, relative_path)

print("ZIP created:", zip_path)
print("Size: {:.2f} MB".format(os.path.getsize(zip_path) / (1024 * 1024)))

from google.colab import files
files.download(zip_path)

ZIP created: /content/judgebert_lmp_submission.zip
Size: 0.06 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [85]:
import json
from pathlib import Path

model_dir = Path("/content/judgebert_lmp_final")
config_path = model_dir / "config.json"

if config_path.exists():
    config = json.loads(config_path.read_text())
    for key in [
        "model_type",
        "architectures",
        "num_labels",
        "hidden_size",
        "max_position_embeddings"
    ]:
        print(f"{key}: {config.get(key)}")
else:
    print("Model config not found.")

model_type: xlm-roberta
architectures: ['XLMRobertaForSequenceClassification']
num_labels: None
hidden_size: 768
max_position_embeddings: 514


In [86]:
from pathlib import Path

dataset_path = Path("/content/lmp_final - lmp_semantic_review(1).csv")

print("Dataset exists:", dataset_path.exists())
if dataset_path.exists():
    print("Dataset size (MB):", round(dataset_path.stat().st_size / (1024 * 1024), 2))

Dataset exists: True
Dataset size (MB): 0.2


In [88]:
import pandas as pd

dataset_path = "/content/lmp_final - lmp_semantic_review(1).csv"
df = pd.read_csv(dataset_path)

# Normalize the target column name for inspection
df["LMP score"] = pd.to_numeric(df["LMP score"], errors="coerce")

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

print("\nLanguage counts:")
print(df["Language"].value_counts(dropna=False))

print("\nLMP score summary:")
print(df["LMP score"].describe())

print("\nMissing values:")
print(df[["Original", "Simplified", "Language", "Domain", "LMP score"]].isna().sum())

pred_path = "/content/lmp_prediction_results.csv"
predictions = pd.read_csv(pred_path)

print("\nPrediction shape:", predictions.shape)
print("Prediction columns:", predictions.columns.tolist())

Dataset shape: (357, 8)

Columns: ['ID', 'Original', 'Language', 'Domain', 'Source', 'Simplified', 'LMP score', 'Human_Assessment']

Language counts:
Language
en    205
hi    152
Name: count, dtype: int64

LMP score summary:
count    357.000000
mean       8.703081
std        1.332952
min        4.000000
25%        8.000000
50%        9.000000
75%       10.000000
max       10.000000
Name: LMP score, dtype: float64

Missing values:
Original      0
Simplified    0
Language      0
Domain        0
LMP score     0
dtype: int64

Prediction shape: (357, 8)
Prediction columns: ['ID', 'Original', 'Simplified', 'Language', 'Domain', 'LMP', 'Predicted_LMP', 'Preservation_Decision']


In [89]:

import json
from pathlib import Path

project_dir = Path("/content/judgebert_lmp_submission")
notebook_dir = project_dir / "notebooks"
notebook_dir.mkdir(parents=True, exist_ok=True)

def markdown_cell(source):
    return {
        "cell_type": "markdown",
        "metadata": {},
        "source": source.splitlines(keepends=True)
    }

def code_cell(source):
    return {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {},
        "outputs": [],
        "source": source.splitlines(keepends=True)
    }

cells = [
    markdown_cell("""# Explainable Multilingual Legal Meaning Preservation

## JUDGEBERT-inspired XLM-RoBERTa Regression

This notebook documents the verified outputs of an academic project evaluating whether simplified legal text preserves the meaning of the original in English and Hindi.

**Reproducibility note:** The original experiment notebook was not found in the current Colab runtime or the inspected Drive location. This notebook is a reconstructed analysis notebook for the saved results. It does not claim to reproduce the original training run exactly.

The model is JUDGEBERT-inspired, not an exact reproduction of the original JUDGEBERT implementation.
"""),
    markdown_cell("""## 1. Dataset and evaluation setup

The supplied dataset contains 357 sentence pairs in English and Hindi. Its target column is `LMP score`, with ratings from 1 to 10. The dataset's assessment provenance should be verified before describing the ratings as independently validated human annotations.

The reported clean baseline comparison uses a common reconstructed split: 242 training, 43 validation, and 72 test rows. Baseline fitting was restricted to training data. See `results/lmp_split_audit.csv` for the split audit.
"""),
    code_cell("""from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/judgebert_lmp_submission")
RESULTS = ROOT / "results"
DATA_PATH = Path("/content/lmp_final - lmp_semantic_review(1).csv")

df = pd.read_csv(DATA_PATH)
df["LMP score"] = pd.to_numeric(df["LMP score"], errors="raise")

print("Dataset shape:", df.shape)
print("Languages:")
display(df["Language"].value_counts().rename_axis("Language").to_frame("Count"))
print("LMP score summary:")
display(df["LMP score"].describe().to_frame("LMP score"))
"""),
    markdown_cell("""## 2. Clean baseline comparison

The following metrics are read from the saved comparison CSV. They are descriptive test-set results; no statistical significance testing was performed.
"""),
    code_cell("""comparison = pd.read_csv(RESULTS / "lmp_model_comparison_clean.csv")
display(comparison)
"""),
    markdown_cell("""## 3. Cross-domain generalization

Two experiments evaluate each model on a domain not used for its training or validation data. Performance is weak in the rental-to-terms/privacy direction.
"""),
    code_cell("""held_out = pd.read_csv(RESULTS / "held_out_domain_results.csv")
display(held_out)
"""),
    markdown_cell("""## 4. Identical and unrelated sentence sanity checks

The saved checks show that XLM-RoBERTa assigns high scores to some unrelated sentence pairs. This is a major limitation: predicted scores must not be interpreted as proof of legal equivalence.
"""),
    code_cell("""sanity = pd.read_csv(RESULTS / "lmp_sanity_check_results.csv")
display(sanity)
"""),
    markdown_cell("""## 5. Qualitative legal-error analysis

Fifteen manually reviewed examples cover possible omissions, scope changes, conditions, retention periods, technical details, and terminology changes. These assessments are preliminary and are not legal advice.
"""),
    code_cell("""qualitative = pd.read_csv(RESULTS / "lmp_qualitative_case_analysis.csv")
print("Cases:", len(qualitative))
print("Missing values per column:")
display(qualitative.isna().sum().to_frame("Missing"))
display(qualitative)
"""),
    markdown_cell("""## 6. Prediction output

The full prediction file contains scores and provisional preservation decisions. The threshold of 8.0 is not independently validated and should be treated as a review heuristic.
"""),
    code_cell("""predictions = pd.read_csv(RESULTS / "lmp_prediction_results.csv")
display(predictions.head())
print("Prediction rows:", len(predictions))
print("Decision counts:")
display(predictions["Preservation_Decision"].value_counts().to_frame("Count"))
"""),
    markdown_cell("""## 7. Limitations and conclusions

- The dataset is small and its ratings are concentrated toward the high end.
- TF-IDF had the strongest Pearson/Spearman correlations and lowest MAE among the reported clean test baselines; MiniLM had the lowest RMSE.
- XLM-RoBERTa did not outperform the baselines on these metrics.
- The transformer assigned high scores to some unrelated pairs.
- Cross-domain generalization was weak.
- Qualitative review and the threshold-based decision are preliminary.
- Legal meaning preservation requires qualified human review, especially for obligations, exceptions, conditions, dates, amounts, entities, and rights.

This notebook analyzes saved experiment artifacts. It does not independently retrain the models or prove that rerunning training will reproduce the recorded metrics.
""")
]

notebook = {
    "cells": cells,
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {"name": "python"}
    },
    "nbformat": 4,
    "nbformat_minor": 5
}

out = notebook_dir / "LMP_results_analysis.ipynb"
out.write_text(json.dumps(notebook, indent=2, ensure_ascii=False), encoding="utf-8")
print("Created:", out)
print("Notebook cells:", len(cells))

Created: /content/judgebert_lmp_submission/notebooks/LMP_results_analysis.ipynb
Notebook cells: 14


In [90]:
from pathlib import Path
import json

root = Path("/content/judgebert_lmp_submission")
nb_path = root / "notebooks" / "LMP_results_analysis.ipynb"

nb = json.loads(nb_path.read_text(encoding="utf-8"))

print("Notebook exists:", nb_path.exists())
print("Notebook cells:", len(nb["cells"]))
print("Result CSVs:", len(list((root / "results").glob("*.csv"))))
print("README exists:", (root / "README.md").exists())

Notebook exists: True
Notebook cells: 14
Result CSVs: 11
README exists: True


In [91]:
import os

print("Current working directory:", os.getcwd())

from google.colab import drive
print("\nChecking common notebook locations:")

for root in [
    "/content/drive/MyDrive/Colab Notebooks",
    "/content"
]:
    if os.path.isdir(root):
        for name in os.listdir(root):
            if "LMP_Baseline_Experiments" in name:
                print(os.path.join(root, name))

Current working directory: /content

Checking common notebook locations:
/content/drive/MyDrive/Colab Notebooks/LMP_Baseline_Experimentsipynb
